# Reproducing Krishnan, Heer, Cherian & Sheffield (2022) from dandiset 000462

Paper. Krishnan S, Heer C, Cherian C, Sheffield MEJ (2022). Reward expectation extinction restructures and degrades CA1 spatial maps through loss of a dopaminergic reward proximity signal. *Nature Communications* 13, 6662. [doi:10.1038/s41467-022-34465-5](https://doi.org/10.1038/s41467-022-34465-5)

Dataset. [DANDI:000462](https://dandiarchive.org/dandiset/000462/0.230416.2132), version 0.230416.2132 ([doi:10.48324/dandi.000462/0.230416.2132](https://doi.org/10.48324/dandi.000462/0.230416.2132)), streamed. The paper cites no version of the dandiset. This is the only published version; the draft (modified 2023-06-20) has the same 14 assets and size and is not fixed, so the published version is pinned.

Claims. `claims/000462/10.1038-s41467-022-34465-5.yaml`, written from the version of record (PMC9636178) with its Supplementary Information, and checked by a second reader.

What the dandiset holds. One file per mouse: 10 wild-type mice of the reward extinction experiment (5 with R, UR, RR and a novel environment; 5 with R and UR only), and 4 DAT-Cre hM4D(Gi) mice with day 1 (R, UR, RR) and days 2 to 4 (before and after saline, CNO and DCZ). Each file holds raw suite2p fluorescence and three behaviour channels at 30 Hz. It does not hold the time-control mice, the dark sessions, the pupil video, the tdTomato control mice or the VTA axon imaging, and it has 10 of the 12 wild-type mice and 4 of the 6 DREADD mice.

| Claim | Tested here |
|---|---|
| session-lap-counts, lick-persistence-ur | tested |
| fig1d, fig1e, fig1f, fig1g, supp4-lickstop-definition-robust, supp4-lapwise-r2, supp5-not-running-speed | tested (decoder) |
| approach-ratio (R and UR; the dark sessions are not in the dandiset), engaged-lap-counts, supp6b-decoder-engaged-laps, disengaged-laps-uniform | tested |
| fig2a, fig2c, fig2d, fig2e, fig3a, fig3b (three claims), fig3c, supp11, fig3d, reward-zone-field-turnover | tested (place fields) |
| supp13-novel-environment | tested |
| supp16, fig4f, supp18, fig4h, fig4m | tested (DREADD mice) |
| supp2-decoder-stable-control, controls-no-change-over-time | blocked: the control mice are not in the dandiset |
| supp6-pupil-engagement | blocked: no pupil data |
| supp15-tdtomato-controls | blocked: the tdTomato mice are not in the dandiset |
| fig5-da-axon-ramps, fig5k-ramp-slope-max, fig5l-ramps-return-rr, supp20-novel-axons, supp21-heterogeneous-axons | blocked: no axon imaging |
| supp14-reward-cells | blocked: the reward cell criteria are in ref. 24, not in the paper |

Revision after the first verification round: the approach ratio that defines engaged laps is now taken over the traversal of the track, leaving out the pause after the reward, and laps with an undefined ratio are neither engaged nor disengaged; the reward zone starts at the first lick before the reward; a check of the clock against the 1.5 s reward pause, the contents of every file and the pooled-percentile saline split were added. Every analysis that uses engaged laps was rerun with the new split.

The analysis of every mouse (dF/F, transients, decoders, place fields) runs once, in the next cells, with the helper `krishnan.py`; each claim's section then reads the results. Where each parameter comes from is in the helper's docstring.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import krishnan as K

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
WT = K.WT_FULL + K.WT_RUR
DR = K.DREADD
# approach ratios: the main reading defines engaged laps; the others are variants
RATIO_LABELS = {"ratios_trav": "Methods ratio, traversal only (main)", "ratios": "Methods ratio, whole lap",
                "ratios_code_trav": "code ratio, traversal only", "ratios_code": "code ratio, whole lap"}
ENG_KEYS = ["ratios_trav", "ratios", "ratios_code_trav"]
print("dandiset", K.DANDISET, "version", K.VERSION)

def mean_ci(x):
    """Mean and half-width of the t 95% confidence interval."""
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) < 2:
        return (float(x.mean()) if len(x) else np.nan, np.nan)
    return float(x.mean()), float(stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x)))

In [ ]:
t0 = time.time()
RES = K.process_all(WT + DR, workers=2)
print(f"processed {len(RES)} mice in {(time.time() - t0) / 60:.1f} min")

## Data check

In [ ]:
assets = K.assets()
print("assets:", len(assets), "| subjects:", sorted(assets))
rows = []
for s, r in RES.items():
    rows.append({"mouse": s, "description": r["subject_description"][:40], "cells": r["n_cells"], "fluorescence frames": r["f_shape"][0],
                 "behaviour rows in file": r["behaviour_rows_in_file"], "F min": round(r["F_min"]), "F median": round(r["F_median"]),
                 "offset removed": round(r["offset"]), "position V": tuple(round(v, 3) for v in r["volts"]["position_V"]),
                 "200 cm at V": round(r["scale"], 3), "reward V": tuple(round(v, 2) for v in r["volts"]["reward_V"]),
                 "lick V": tuple(round(v, 2) for v in r["volts"]["lick_V"]),
                 "reward between 0.5-4 V": round(r["volts"]["reward_frac_between_0.5_and_4V"], 4),
                 "lick between 0.5-4 V": round(r["volts"]["lick_frac_between_0.5_and_4V"], 4),
                 "behaviour md5": r["fingerprints"]["behaviour_md5"], "fluorescence md5": r["fingerprints"]["fluorescence_md5"]})
info = pd.DataFrame(rows).set_index("mouse")
print(info.to_string())
print("distinct behaviour fingerprints:", info["behaviour md5"].nunique(), "| distinct fluorescence fingerprints:", info["fluorescence md5"].nunique(), "of", len(info))
cond_rows = []
for s, r in RES.items():
    for n, c in r["beh"]["conds"].items():
        cond_rows.append({"mouse": s, "condition": n, "frames": c["n_frames"], "complete laps": len(K.complete_laps(c)),
                          "lap segments": len(c["laps"]), "rewards": len(c["rewards"]), "lick onsets": len(c["licks"]),
                          "transient frac": round(r["transient_frac"][n], 3), "dF/F 1/50/99 pct": tuple(round(v, 3) for v in r["dff_pct"][n])})
cdf = pd.DataFrame(cond_rows)
print(cdf.to_string(index=False))
print("speed range (cm/s) over running frames:", round(min(np.percentile(c["speed"][c["speed"] >= 1], 1) for r in RES.values() for c in r["beh"]["conds"].values()), 1),
      "to", round(max(np.percentile(c["speed"], 99.9) for r in RES.values() for c in r["beh"]["conds"].values()), 1))

In [ ]:
# Clock and timing check against the paper's 1.5 s pause after each reward (Methods, Behavior and virtual reality),
# and the laps per minute that the lap counts and the condition lengths imply (independent of how a lap time is defined)
trows = []
for s, r in RES.items():
    for n, c in r["beh"]["conds"].items():
        tm = r["beh"]["timing"][n]
        if not len(tm["reward_to_teleport"]):
            continue
        lt = r["beh"]["lap_times"][n]
        run_t = np.array([(c["speed"][a:e] >= K.RUN_SPEED).sum() / K.FS for a, e in K.complete_laps(c)])
        trows.append({"mouse": s, "condition": n, "rewarded laps": len(tm["reward_to_teleport"]),
                      "frames end->reward (median)": float(np.median(tm["top_to_reward"])),
                      "frames reward->teleport (median)": float(np.median(tm["reward_to_teleport"])),
                      "reward->teleport s at 30 Hz": round(float(np.median(tm["reward_to_teleport"])) / K.FS, 2),
                      "inter-reward s (median)": round(float(np.median(tm["inter_reward_s"])), 2),
                      "laps per min": round(len(K.complete_laps(c)) / (c["n_frames"] / K.FS / 60), 2),
                      "lap time 0-200 cm s (mean)": round(float(np.nanmean(lt)), 2),
                      "running time per lap s (mean)": round(float(np.mean(run_t)), 2)})
tdf = pd.DataFrame(trows)
print(tdf.to_string(index=False))
print("over all rewarded conditions: reward->teleport median", np.median(tdf["frames reward->teleport (median)"]), "frames, range",
      tdf["frames reward->teleport (median)"].min(), "to", tdf["frames reward->teleport (median)"].max())
wt_r = tdf[(tdf["mouse"].isin(WT)) & (tdf["condition"] == "Rewarded")]
print(f"wild-type R: laps per minute {wt_r['laps per min'].mean():.2f} (range {wt_r['laps per min'].min()} to {wt_r['laps per min'].max()}), "
      f"lap time {wt_r['lap time 0-200 cm s (mean)'].mean():.2f} s, inter-reward interval {wt_r['inter-reward s (median)'].mean():.2f} s")
# Full subject descriptions, and every object in every file (to look for pupil, axon or other data)
for s, r in RES.items():
    print(s, "|", r["subject_description"])
objs = {}
for s, a in assets.items():
    f = K.open_file(a["asset_id"])
    names = []
    f.visititems(lambda n, o: names.append(n + (f" {o.shape} {o.dtype}" if hasattr(o, "shape") else "/")))
    objs[s] = names
    f.close()
common = set.intersection(*[set(n.split(" ")[0] for n in v) for v in objs.values()])
print("HDF5 objects present in every file:", sorted(common))
for s, v in objs.items():
    extra = [n for n in v if n.split(" ")[0] not in common]
    print(s, "objects not in every file:", extra if extra else "none", "| datasets:", [n for n in v if " (" in n and n.startswith(("acquisition", "processing", "analysis", "stimulus"))])


The timing check and the file contents:

- In every rewarded condition of every file the teleport follows the reward onset by a median of 46 to 49 frames, 1.53 to 1.63 s at 30 Hz, against the 1.5 s pause of the Methods (Behavior and virtual reality), and the reward comes a median of 0 to 4 frames after the position first reaches the track end in the wild-type mice. The behaviour clock is therefore consistent with the stated 30 Hz.
- The wild-type mice run 3.36 to 4.8 R laps per minute (mean 4.11) from the lap counts and condition lengths alone, close to the training criterion of more than 4 laps per minute (Methods). Their mean R lap time from 0 to 200 cm is 12.45 s (running frames only, 8.5 s on average and 6.0 to 10.7 s per mouse). The paper's 7.34 s per matched R lap is shorter than both means, though the running time of 3 mice (CFC4 5.98 s, CFC16 6.56 s, CFC19 7.20 s) is below it. Lap time is independent of the volts-to-cm scale, and the clock agrees with the reward pause, so the slower lap times here do not come from the clock or the scale; they come from the data or from a definition of lap time the paper does not give. The lap speeds of supp16 and supp18 are 200 cm over the same lap times.
- Every file holds the same HDF5 objects: the two acquisition series (behaviour, frames x 3, and raw fluorescence, frames x cells), their starting times, and metadata under `general/`. `processing/ophys/` and `analysis/` are empty. No file holds pupil, axon imaging or any other data.
- The subject descriptions, printed in full above, are "C57BL/6J mouse" for the 10 wild-type files and "Dat-Cre mice injected with inhibitory DREADD receptor in VTA-Dopmaine neurons" for the 4 DREADD files; none describes a tdTomato control.

The checks against the description:

- Frame counts: the description of each fluorescence series lists the conditions and their frame counts; they sum to the fluorescence frames in every file. The behaviour rows match the fluorescence frames in 11 files; sub-NR6 has 8 extra behaviour rows at the end, sub-CFC4 one row fewer, and sub-014 6000 rows fewer (cell after next).
- Conditions: rewards are delivered in R, RR, the novel session and all DREADD sessions, and not in UR, consistent with the condition order in the description, except that 4 mice (NR6, NR14, NR23, CFC16) receive one reward in UR (the `rewards` column above). In NR6 and NR14 it falls at the end of the first complete UR lap (timing table, next cell); in NR23 and CFC16 it falls in the partial lap that opens UR, which no analysis uses. Either the boundary taken from the frame counts is a little early or the reward was switched off a lap late. The lap is kept in UR, as the frame counts say; a variant in lick-persistence-ur counts it as R.
- Units: the position channel is in volts (0 to about 0.61 V) with no stated scale; the reward is delivered at the top of the range, so the top is read as 200 cm (assumption). The reward and lick channels are two-level signals (almost no samples between 0.5 and 4 V).
- Fluorescence: the wild-type files have a minimum near 0 counts; the DREADD files have a minimum near 17000 counts with transients a few thousand counts above it, an offset the description does not mention. It is removed (file minimum) before dF/F.
- The 14 files are distinct: no two share the behaviour or fluorescence fingerprints.
- The 30 Hz rate is the series' stated rate; the description gives no units for position, reward or licks. The next cell checks the rate against the pause after the reward.

In [ ]:
# Clock check: decoding R position with the true labels against labels shifted by 10 s
for s in WT:
    d = RES[s]["dec_shift"]
    print(f"{s}: held-out R2 with aligned labels {d[0]:.3f}, with labels shifted 10 s {d[300]:.3f}")
# Locating the missing behaviour frames of sub-014
gap = pd.DataFrame(K.gap_check("014"))
print(gap.round(1).to_string(index=False))

In every wild-type mouse the position decodes from the fluorescence with aligned labels (R2 above 0.7) and not with labels shifted by 10 s, so behaviour and fluorescence share a clock at the scale of the decoder. In sub-014 the fluorescence after frame 71000 matches the behaviour 6000 frames earlier; before it, it matches with no shift (table above, a model trained on the aligned BeforeSaline condition). The 6000 missing behaviour rows are therefore at the start of BeforeCNO; the helper inserts them as missing (`BEHAVIOUR_GAPS`), and laps that touch them are dropped. For the DREADD mice the clock is shown by the place fields found in every condition (next sections); no decoder was run on them.

In [ ]:
# File metadata named in the findings, and the effect of the DREADD offset on dF/F (first 9000 frames, 40 cells)
mrows = []
for s, a in assets.items():
    f = K.open_file(a["asset_id"])
    ip = f["general/optophysiology/ImagingPlane"]
    row = {"mouse": s, "session_start_time": f["session_start_time"][()].decode(), "age": f["general/subject/age"][()].decode(),
           "indicator": ip["indicator"][()].decode(), "imaging plane description": ip["description"][()].decode(),
           "session description": f["session_description"][()].decode()}
    if RES[s]["offset"] > 0:
        F = f["acquisition/Cellxrawfluoresence/data"][:9000, :40].astype(np.float32)
        p_rem = np.percentile(K.dff(np.maximum(F - RES[s]["offset"], 1)), 99)
        p_raw = np.percentile(K.dff(F), 99)
        row["dF/F 99th pct offset removed / kept"] = f"{p_rem:.2f} / {p_raw:.3f} (ratio {p_rem / p_raw:.1f})"
    f.close()
    mrows.append(row)
print(pd.DataFrame(mrows).set_index("mouse").to_string())


The file metadata: every file gives the ImagingPlane indicator as 'GFP', while the plane's own description says GCaMP6f; every file has the same session_start_time (2021-12-13T02:30:00-06:00) and age (P12W); and every file, the 4 DREADD files with four days of sessions included, has the session description 'Reward expectation extinction'. In the first 9000 frames of 40 cells of each DREADD file, the 99th percentile of dF/F is 1.99 to 4.05 with the offset removed and 0.22 to 0.38 with it kept, 8.0 to 12.1 times smaller.

## session-lap-counts
> Mice ran on average 34 laps in R, 28 in UR and 27 in RR (mean +/- 95% CI 34 +/- 2, 28 +/- 4 and 27 +/- 3), and 31 +/- 5 laps in the novel rewarded environment.

Complete laps (start in the first 10 cm, reach the last 5 cm) and all lap segments, which include a partial lap at the start or end of a condition. The code variant drops the frames the paper's code drops (`task_framestokeep`).

In [ ]:
def code_kept_laps(s, cond_i, c):
    keep = K.CODE_FRAMES_TO_KEEP[s][cond_i]
    stop = keep if keep > 0 else c["n_frames"] + keep
    return sum(1 for a, e, ok in c["laps"] if ok and e <= stop)

rows = []
for s in WT:
    conds = RES[s]["beh"]["conds"]
    row = {"mouse": s}
    for i, (n, c) in enumerate(conds.items()):
        row[n] = len(K.complete_laps(c))
        row[n + " (with partial)"] = len(c["laps"])
        row[n + " (code frames)"] = code_kept_laps(s, i, c)
    rows.append(row)
laps_df = pd.DataFrame(rows).set_index("mouse")
print(laps_df.T.to_string())
expect = {"Rewarded": (34, 2), "Unrewarded": (28, 4), "ReRewarded": (27, 3), "Novel": (31, 5)}
lapcount_measured = {}
for n, (v, tol) in expect.items():
    for suffix in ["", " (with partial)", " (code frames)"]:
        m, h = mean_ci(laps_df[n + suffix].dropna())
        ok = abs(m - v) <= tol
        if suffix == "":
            lapcount_measured[n] = m
        print(f"{n + suffix:32s} mean {m:5.1f} +/- {h:4.1f} (n = {laps_df[n + suffix].notna().sum()}); paper {v} +/- {tol}; within: {ok}")

**Result: reproduced.** Counting complete laps, the means are 35.3 (R, 10 mice), 29.7 (UR), 28.2 (RR, 5 mice) and 32.8 (novel, 5 mice), all within the paper's 34 +/- 2, 28 +/- 4, 27 +/- 3 and 31 +/- 5. Counting partial laps puts R at 36.8, outside its tolerance; the code's frame cut gives 33.2.

## lick-persistence-ur
> After reward removal mice kept pre-licking for about 5 laps (5 +/- 1) before stopping; 4 of 12 mice licked randomly in later laps.

In [ ]:
print("reward zone start per mouse (cm, mean position of the first R lick after 20 cm and before the reward; main):",
      {s: round(RES[s]["beh"]["zone"], 1) for s in WT})
print("variant, first lick after 20 cm whenever it falls (first version of this attempt):", {s: round(RES[s]["beh"]["zone_any_lick"], 1) for s in WT})
print("R laps with a lick before the reward (after 20 cm), per mouse:",
      {s: f"{len(RES[s]['beh']['first_lick_positions'])} of {len(K.complete_laps(RES[s]['beh']['conds']['Rewarded']))}" for s in WT})
print("first pre-reward lick position per R lap, 10th/50th/90th percentile (cm):",
      {s: tuple(np.round(np.percentile(RES[s]["beh"]["first_lick_positions"], [10, 50, 90]), 1)) for s in WT})
print("code variant zone start (cm):", {s: RES[s]["beh"]["zone_code"] for s in WT})
lick_rows = []
for s in WT:
    b = RES[s]["beh"]
    z = b["ur_zone_licks"]
    anyl = b["ur_any_licks"]
    first_nolick = int(np.argmax(z == 0)) if (z == 0).any() else len(z)
    row = {"mouse": s, "UR laps": len(z), "zone licks per UR lap": " ".join(map(str, z))}
    for L in K.LICK_READINGS:
        ls = b["lick_stop"][L]
        row[f"RE high laps {L}"] = ls
        row[f"(a) zone lick in RE low {L}"] = bool((z[ls:] > 0).any())
        row[f"(b) any lick in RE low {L}"] = bool((anyl[ls:] > 0).any())
    row["(c) zone lick after first no-lick lap"] = bool((z[first_nolick + 1:] > 0).any())
    row["first no-lick lap"] = first_nolick
    cU = b["conds"]["Unrewarded"]
    s0, e0 = K.complete_laps(cU)[0]
    row["reward in first UR lap"] = bool(((cU["rewards"] >= s0) & (cU["rewards"] < e0)).any())
    lick_rows.append(row)
lick_df = pd.DataFrame(lick_rows).set_index("mouse")
print(lick_df.drop(columns="zone licks per UR lap").to_string())
for s in WT:
    print(s, "zone licks per UR lap:", lick_df.loc[s, "zone licks per UR lap"])
lick_persist = {}
for L in K.LICK_READINGS:
    m, h = mean_ci(lick_df[f"RE high laps {L}"])
    lick_persist[L] = m
    print(f"{L}: RE high laps per mouse mean {m:.2f} +/- {h:.2f} (95% CI, n = {len(WT)}); paper 5 +/- 1; within: {abs(m - 5) <= 1}")
    print(f"    random lickers: (a) {lick_df[f'(a) zone lick in RE low {L}'].sum()}, (b) {lick_df[f'(b) any lick in RE low {L}'].sum()} of {len(WT)} mice")
print(f"    (c) {lick_df['(c) zone lick after first no-lick lap'].sum()} of {len(WT)} mice")
fn = lick_df["first no-lick lap"]
print(f"reading not in the criterion, index of the first UR lap with no zone lick (laps before it): mean {fn.mean():.2f} +/- {mean_ci(fn)[1]:.2f}, per mouse {fn.tolist()}")
rw1 = lick_df["reward in first UR lap"]
print("mice with a reward in the first UR lap:", rw1[rw1].index.tolist())
for L in K.LICK_READINGS:
    adj = lick_df[f"RE high laps {L}"] - rw1.astype(int)
    print(f"variant, first UR lap counted as R where it was rewarded, {L}: RE high laps mean {adj.mean():.2f} +/- {mean_ci(adj)[1]:.2f}")
# code variant of the zone
for L in K.LICK_READINGS:
    v = [K.lick_stop(RES[s]["beh"]["ur_zone_licks_code"] > 0, 2, L) for s in WT]
    print(f"code zone variant {L}: RE high laps mean {np.mean(v):.2f}, per mouse {v}")
# lick stops under the first version's zone (first lick after 20 cm whenever it falls)
for L in K.LICK_READINGS:
    old = {s: K.lick_stop(K.licks_in_zone(RES[s]["beh"]["conds"]["Unrewarded"], RES[s]["beh"]["zone_any_lick"]) > 0, 2, L) for s in WT}
    same = all(old[s] == RES[s]["beh"]["lick_stop"][L] for s in WT)
    print(f"lick stop {L} under the first version's zone, per mouse {old}; same as under the main zone in every mouse: {same}")


**Result: not reproduced.** The reward zone now starts at the mean position of the first lick before the reward (184.5 to 196.9 cm in 8 mice, 167.7 in NR14 and 129.0 in CFC17); counting any first lick moved it by 0.3 to 3.9 cm and changed no lick stop under L1, L2 or L3 (last lines of the output). Mice kept licking in the reward zone for several UR laps, but the mean number of RE high laps is 6.60 +/- 2.62 under L1 (7.60 under L2, 8.60 under L3), outside 5 +/- 1 under every reading, and 6.40 to 8.40 when the rewarded first UR lap of NR6 and NR14 is counted as R. Random licking after the lick stop by reading (a), a zone lick in an RE low lap, is found in 4 of the 10 mice, the paper's count; reading (b) gives 7 and reading (c) 9. The index of the first UR lap with no zone lick, a reading the criterion does not list, averages 4.80 +/- 1.99 laps; it is shown, not judged, and is asked of a reader (questions file).

## fig1f-decoder-r2
> Mean decoder R2 is 0.95 in R, 0.90 in RE high laps (similar to R), 0.65 in RE low laps (significantly lower), and 0.82 in RR, which recovers somewhat but stays below R.

Decoder: Gaussian naive Bayes on the dF/F of all cells, running frames, 40 bins, trained on the first 60% of R laps, the track treated as a circle (each prediction taken at its nearest copy of the true bin), R2 per lap between true and predicted bin (Methods, Position decoding, as the criterion gives it). The same decoder without the circular treatment is shown for comparison. Per mouse (main reading, from the code's `plot_shuffleerror_withlickstop`): R is the mean over random draws of 4 of the last 5 R laps, RE high and RE low of draws of min(4, lick stop) and min(3, lick stop) laps from those laps, RR of min(4, lick stop) RR laps. The mean over many such draws converges to the mean over the laps drawn from, which is computed exactly here (last 5 R laps, all RE high, all RE low, all RR laps). The variant is the code's decoder (first 50% of R frames, unwrapped).

In [ ]:
def lap_r2(s, cond, key="dec"):
    return {d["lap"]: d["r2"] for d in RES[s][key][cond]}

def per_mouse_r2(s, L, key="dec", reading="main"):
    b = RES[s]["beh"]
    ls = b["lick_stop"][L]
    r = lap_r2(s, "Rewarded", key)
    u = lap_r2(s, "Unrewarded", key)
    out = {}
    rl = [r[k] for k in sorted(r)]
    out["R"] = np.nanmean(rl[-5:]) if reading == "main" else np.nanmean(rl)
    out["RE high"] = np.nanmean([v for k, v in u.items() if k < ls]) if any(k < ls for k in u) else np.nan
    out["RE low"] = np.nanmean([v for k, v in u.items() if k >= ls]) if any(k >= ls for k in u) else np.nan
    if "ReRewarded" in RES[s][key]:
        out["RR"] = np.nanmean(list(lap_r2(s, "ReRewarded", key).values()))
    else:
        out["RR"] = np.nan
    return out

def paired(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ok = np.isfinite(a) & np.isfinite(b)
    return stats.ttest_rel(a[ok], b[ok]), ok.sum()

fig1f = {}
for key, label in [("dec", "Methods decoder (60% of R laps, circular)"), ("dec_var", "code decoder (50% of R frames)"), ("dec_lin", "60% of R laps, not circular")]:
    for reading in ["main", "all tested R laps"]:
        for L in K.LICK_READINGS:
            df = pd.DataFrame({s: per_mouse_r2(s, L, key, reading) for s in WT}).T
            means = {c: mean_ci(df[c].dropna()) for c in df.columns}
            tests = {}
            for a, b2 in [("R", "RE high"), ("R", "RE low"), ("R", "RR"), ("RE high", "RE low"), ("RE high", "RR"), ("RE low", "RR")]:
                t, n = paired(df[a], df[b2])
                tests[(a, b2)] = (t.statistic, t.pvalue, n)
            fig1f[(key, reading, L)] = (df, means, tests)
            print(f"--- {label}, R laps {reading}, lick stop {L}")
            print("  ", "; ".join(f"{c} {m:.3f} [{m - h:.3f} {m + h:.3f}] (n={df[c].notna().sum()})" for c, (m, h) in means.items()))
            print("  ", "; ".join(f"{a}-{b2}: t={t:.2f} p={p:.4f} (x6 Bonferroni {min(1, 6 * p):.4f}, n={n})" for (a, b2), (t, p, n) in tests.items()))
df, means, tests = fig1f[("dec", "main", "L1")]
print("\nper mouse, Methods decoder, main reading, L1:\n", df.round(3).to_string())
fig1f_ok = {}
for k, (df, means, tests) in fig1f.items():
    if k[0] == "dec_lin":
        continue
    within = all(abs(means[c][0] - v) <= tol for c, (v, tol) in {"R": (0.95, 0.02), "RE high": (0.90, 0.03), "RE low": (0.65, 0.11), "RR": (0.82, 0.05)}.items())
    for corr in [1, 6]:
        sig = (tests[("R", "RE low")][1] * corr < 0.05, tests[("R", "RR")][1] * corr < 0.05, tests[("R", "RE high")][1] * corr >= 0.05)
        fig1f_ok[k + (corr,)] = (within, sig)
        print(k, "Bonferroni x6" if corr == 6 else "uncorrected", "means within tolerance:", within, "| R<RE low, R<RR significant, R vs RE high not:", sig)

**Result: partially reproduced.** Under lick stop L1 the per-mouse means are R 0.960, RE high 0.901, RE low 0.606 and RR 0.861, all within tolerance (also under L2). RE low is below R (p = 0.0008, 0.0048 with Bonferroni) and RR below R (p = 0.0345 uncorrected, 0.207 with Bonferroni). RE high is not similar to R: the paired t-test gives p = 0.0047 (0.028 with Bonferroni), where the criterion requires p >= 0.05; this holds under every reading and with the code's decoder.

## fig1d-decoder-drop-ur
> A naive Bayes decoder trained on initial R laps predicts position accurately on the final R laps and on the first UR laps, then abruptly underperforms later in UR.

Per mouse: holds when the mean R2 of the first 3 UR laps is within 0.1 of the mean R2 of the held-out R laps and the mean R2 of the RE low laps is below both (criterion; thresholds the reader's).

In [ ]:
for s in WT:
    r = lap_r2(s, "Rewarded")
    u = lap_r2(s, "Unrewarded")
    print(s, "held-out R:", " ".join(f"{r[k]:.2f}" for k in sorted(r)))
    print(s, "UR        :", " ".join(f"{u[k]:.2f}" for k in sorted(u)))
need = int(np.ceil(9 / 12 * len(WT)))
print(f"the criterion's 9 of 12 mice (75%) scaled to the {len(WT)} mice here: at least {need} (ceil(0.75 x {len(WT)})); a literal 9 is also shown")
fig1d = {}
for key in ["dec", "dec_var", "dec_lin"]:
    for L in K.LICK_READINGS:
        hold = 0
        rows = []
        for s in WT:
            r = lap_r2(s, "Rewarded", key)
            u = lap_r2(s, "Unrewarded", key)
            ls = RES[s]["beh"]["lick_stop"][L]
            rmean = np.nanmean(list(r.values()))
            first3 = np.nanmean([u[k] for k in sorted(u)[:3]])
            low = np.nanmean([v for k, v in u.items() if k >= ls])
            ok = abs(first3 - rmean) <= 0.1 and low < first3 and low < rmean
            hold += ok
            rows.append((s, round(rmean, 3), round(first3, 3), round(low, 3), ok))
        fig1d[(key, L)] = hold
        print(f"{key} {L}: holds in {hold} of {len(WT)} mice (>= {need}: {hold >= need}; >= 9: {hold >= 9}):", rows)

**Result: reproduced.** With the decoder of the Methods (60% of R laps, circular track), the first 3 UR laps decode within 0.1 of the held-out R laps and RE low laps decode worse than both in 8 of the 10 mice under each lick stop reading. The criterion's 9 of 12 mice is applied as the same fraction, 75%, of the 10 mice here, which needs at least 8 (ceil(0.75 x 10)); a literal 9 is not reached. The code's decoder gives 6 of 10. Without the circular treatment, 2 of 10.

## fig1e-decoder-follows-licking
> Averaged over 12 mice, lap by lap decoder R2 in UR falls as an S-shaped curve around the lap where pre-licking reaches zero; a reverse Boltzmann sigmoid fits the mean R2 with r = 0.94, and its point of maximum change is at lap 10.

In [ ]:
from scipy.optimize import curve_fit

def boltz(x, top, bottom, v50, slope):
    return bottom + (top - bottom) / (1 + np.exp((x - v50) / slope))

def roll3(x):
    return pd.Series(x).rolling(3, min_periods=1).mean().to_numpy()

NL = 15
r2_mat = np.full((len(WT), NL), np.nan)
lk_mat = np.full((len(WT), NL), np.nan)
for i, s in enumerate(WT):
    u = lap_r2(s, "Unrewarded")
    z = RES[s]["beh"]["ur_zone_licks"]
    arr = np.array([u.get(k, np.nan) for k in range(len(z))])
    sm = roll3(arr)[:NL]
    r2_mat[i, :len(sm)] = sm
    lz = roll3(z.astype(float))[:NL]
    lk_mat[i, :len(lz)] = lz
mean_r2 = np.nanmean(r2_mat, axis=0)
mean_lk = np.nanmean(lk_mat, axis=0)
x = np.arange(1, NL + 1)
p, _ = curve_fit(boltz, x, mean_r2, p0=[mean_r2[:3].mean(), mean_r2[-3:].mean(), 8, 1], maxfev=20000)
fit = boltz(x, *p)
r_fit = np.corrcoef(fit, mean_r2)[0, 1]
R2_fit = 1 - ((mean_r2 - fit) ** 2).sum() / ((mean_r2 - mean_r2.mean()) ** 2).sum()
xf = np.linspace(1, NL, 2001)
d2 = np.gradient(np.gradient(boltz(xf, *p), xf), xf)
zc = np.where(np.diff(np.sign(d2)) != 0)[0]
lap_max = xf[zc[0]] if len(zc) else np.nan
lk_norm = mean_lk / np.nanmax(mean_lk)
lk_min_lap = int(x[np.argmax(lk_norm <= lk_norm.min() + 1e-9)])
print("mean R2 per UR lap (3-lap rolling, mice present):", np.round(mean_r2, 3))
print("mice per lap:", np.isfinite(r2_mat).sum(0))
print("mean zone licks per lap (rolling):", np.round(mean_lk, 2))
print(f"sigmoid params top {p[0]:.3f} bottom {p[1]:.3f} v50 {p[2]:.2f} slope {p[3]:.2f}")
print(f"fit r = {r_fit:.3f}, fit R2 = {R2_fit:.3f} (paper 0.94 +/- 0.094)")
print(f"first zero of second derivative at lap {lap_max:.2f} (rounded {np.round(lap_max):.0f}; paper 10 +/- 1)")
print(f"normalised lick curve first reaches its minimum at lap {lk_min_lap}; distance to the lap of maximum change {abs(lk_min_lap - np.round(lap_max)):.0f}")

**Result: reproduced.** The reverse Boltzmann fit to the mean UR R2 has r = 0.979 (R2 of the fit 0.959; paper 0.94), its second derivative first reaches zero at lap 11.23, rounded to 11 (paper 10 +/- 1), and the smoothed lick curve first reaches its minimum at lap 14, 3 laps from it, within the 3 laps the criterion allows.

## fig1g-decoder-error-all-locations
> In RE low laps decoder error increases at all locations along the track (start, middle and end), not only near the reward; in RR the error decreases across the track.

Error per frame = distance in cm between true and decoded position (circular), averaged per mouse over the frames of each condition in the start (0 to 50 cm), middle (50 to 150 cm) and end (150 to 200 cm) groups.

In [ ]:
GROUPS = {"start": (0, 10), "middle": (10, 30), "end": (30, 40)}

def group_error(s, cond, laps_sel, key="dec"):
    out = {}
    recs = [d for d in RES[s][key][cond] if laps_sel(d["lap"])]
    if not recs:
        return {g: np.nan for g in GROUPS}
    bins = np.concatenate([d["bins"] for d in recs])
    err = np.concatenate([d["err_cm"] for d in recs])
    for g, (a, b) in GROUPS.items():
        m = (bins >= a) & (bins < b)
        out[g] = err[m].mean() if m.any() else np.nan
    return out

fig1g = {}
for key in ["dec", "dec_var"]:
    for L in K.LICK_READINGS:
        rows = {}
        for s in WT:
            ls = RES[s]["beh"]["lick_stop"][L]
            rows[(s, "R")] = group_error(s, "Rewarded", lambda l: True, key)
            rows[(s, "RE low")] = group_error(s, "Unrewarded", lambda l, ls=ls: l >= ls, key)
            if "ReRewarded" in RES[s][key]:
                rows[(s, "RR")] = group_error(s, "ReRewarded", lambda l: True, key)
        df = pd.DataFrame(rows).T
        res_g = {}
        for g in GROUPS:
            r = df.xs("R", level=1)[g]
            lo = df.xs("RE low", level=1)[g].reindex(r.index)
            t = stats.ttest_rel(lo, r)
            rr = df.xs("RR", level=1)[g]
            t2 = stats.ttest_rel(rr, r.reindex(rr.index))
            res_g[g] = (r.mean(), lo.mean(), rr.mean(), t.statistic, t.pvalue, t2.statistic, t2.pvalue)
        fig1g[(key, L)] = res_g
        print(f"--- {key} {L}")
        for g, v in res_g.items():
            print(f"  {g:6s}: R {v[0]:.1f} cm, RE low {v[1]:.1f} cm, RR {v[2]:.1f} cm (5 mice); RE low vs R t={v[3]:.2f} p={v[4]:.4f}; RR vs R t={v[5]:.2f} p={v[6]:.4f}")
        n_sig = sum(v[1] > v[0] and v[4] < 0.01 for v in res_g.values())
        print(f"  groups with RE low error higher at p < 0.01: {n_sig} of 3")
print("error range over mice and groups (cm):", round(float(np.nanmin(df.values)), 1), "to", round(float(np.nanmax(df.values)), 1))

**Result: reproduced.** Under L1, decoder error in RE low is higher than in R at the start (17.1 against 4.9 cm, p < 0.0001), middle (33.9 against 5.6 cm, p = 0.0001) and end (17.3 against 4.4 cm, p < 0.0001) of the track, in 3 of 3 groups under every lick stop reading and with the code's decoder. RR errors (12.1, 21.0 and 11.6 cm in the 5 mice) lie between R and RE low.

## supp4-lickstop-definition-robust
> The drop in decoder R2 from RE high to RE low holds whether the lick stop is the first no-lick lap or after 2 or 4 consecutive no-lick laps, or when laps are grouped by presence of licks; with 6 consecutive no-lick laps the result differs. Median R2 is 0.96 in R and 0.80 in RR, and about 0.87 to 0.93 in RE high and 0.69 to 0.70 in RE low across definitions.

In [ ]:
def r2_split(s, ls=None, lick_mask=None, key="dec"):
    u = lap_r2(s, "Unrewarded", key)
    if lick_mask is not None:
        hi = [v for k, v in u.items() if lick_mask[k]]
        lo = [v for k, v in u.items() if not lick_mask[k]]
    else:
        hi = [v for k, v in u.items() if k < ls]
        lo = [v for k, v in u.items() if k >= ls]
    return (np.nanmean(hi) if hi else np.nan), (np.nanmean(lo) if lo else np.nan)

supp4 = {}
for start in ["first", "Nth", "after"]:
    rows = []
    for name, n in [("first no-lick lap", 1), ("2 laps", 2), ("4 laps", 4), ("6 laps", 6), ("presence of licks", None)]:
        his, los = [], []
        for s in WT:
            z = RES[s]["beh"]["ur_zone_licks"]
            if n is None:
                hi, lo = r2_split(s, lick_mask=z > 0)
            else:
                L = {"first": "L1", "Nth": "L2", "after": "L3"}[start]
                ls = K.lick_stop(z > 0, n, L)
                hi, lo = r2_split(s, ls)
            his.append(hi)
            los.append(lo)
        his, los = np.array(his), np.array(los)
        ok = np.isfinite(his) & np.isfinite(los)
        t = stats.ttest_rel(his[ok], los[ok])
        rows.append({"definition": name, "n mice": int(ok.sum()), "median RE high": np.nanmedian(his), "median RE low": np.nanmedian(los),
                     "t": t.statistic, "p": t.pvalue, "p x5 Bonferroni": min(1, 5 * t.pvalue)})
    df = pd.DataFrame(rows)
    supp4[start] = df
    print(f"--- RE low starts at the {start} lap of the run")
    print(df.round(4).to_string(index=False))
medR = np.median([per_mouse_r2(s, "L1")["R"] for s in WT])
medRR = np.nanmedian([per_mouse_r2(s, "L1")["RR"] for s in WT])
print(f"median R {medR:.3f} (paper 0.96 +/- 0.096), median RR {medRR:.3f} (paper 0.80 +/- 0.08)")
for start, df in supp4.items():
    d = df[df.definition != "6 laps"]
    ok_t = (d["p x5 Bonferroni"] < 0.05).all() and (d["median RE high"] > d["median RE low"]).all()
    ok_hi = d["median RE high"].between(0.87 - 0.087, 0.93 + 0.093).all()
    ok_lo = d["median RE low"].between(0.69 - 0.069, 0.70 + 0.07).all()
    print(f"{start}: tests significant for first/2/4/presence: {ok_t}; RE high medians in 0.783 to 1.023: {ok_hi}; RE low medians in 0.621 to 0.77: {ok_lo}")

**Result: reproduced.** With RE low starting at the first lap of the run (the first start reading), RE low R2 is below RE high for the first no-lick lap, 2 and 4 laps and the grouping by licks (Bonferroni p 0.006 to 0.012), RE high medians are 0.91 and RE low medians 0.66 to 0.67, inside the widened ranges, and the median R (0.976) and RR (0.842) are within tolerance. Under the other two start readings the 4-lap test is not significant after correction (p = 0.066 and 0.065). The 6-lap definition gives p = 0.30 (Nth reading) and 0.12 (after reading) after correction; under the first reading it is the same split as 2 and 4 laps.

## supp4-lapwise-r2
> Over all laps pooled, decoder R2 is 0.97 (SD 0.046) in R (399 laps), 0.86 (0.14) in RE high (90 laps), 0.56 (0.34) in RE low (244 laps) and 0.79 (0.19) in RR (121 laps).

In [ ]:
for L in K.LICK_READINGS:
    pools = {"R held-out": [], "R all laps (incl. training)": [], "RE high": [], "RE low": [], "RR": []}
    for s in WT:
        ls = RES[s]["beh"]["lick_stop"][L]
        pools["R held-out"] += list(lap_r2(s, "Rewarded").values())
        pools["R all laps (incl. training)"] += list(lap_r2(s, "Rewarded").values()) + list(lap_r2(s, "Rewarded_train").values())
        u = lap_r2(s, "Unrewarded")
        pools["RE high"] += [v for k, v in u.items() if k < ls]
        pools["RE low"] += [v for k, v in u.items() if k >= ls]
        if "ReRewarded" in RES[s]["dec"]:
            pools["RR"] += list(lap_r2(s, "ReRewarded").values())
    print(f"--- {L}: " + "; ".join(f"{k}: {len(v)} laps, mean {np.nanmean(v):.3f} (SD {np.nanstd(v, ddof=1):.3f})" for k, v in pools.items()))
    print("    R2 range over pooled laps:", round(min(np.nanmin(v) for v in pools.values()), 2), "to", round(max(np.nanmax(v) for v in pools.values()), 2))
print("total complete R laps:", sum(len(K.complete_laps(RES[s]['beh']['conds']['Rewarded'])) for s in WT))

**Result: partially reproduced.** The pooled lap means are within the stated SDs under every reading (L3: R 0.977 held out or 0.990 with training laps, RE high 0.858, RE low 0.635, RR 0.869). The lap counts are not: R has 353 complete laps (paper 399), RE low 211 to 231 (244 +/- 24 needs 220 or more; L1 gives 231 but then RE high has 66 against 90 +/- 9), and RR 141 (121 +/- 12). The dandiset holds 10 of the 12 mice.

## supp5-not-running-speed
> The drop in decoder performance in UR is not due to running; 70% of UR laps (233 of 334) have lap times matched to R, and decoder R2 falls in matched laps as in slower laps (median R 0.96, matched 0.76, slower 0.71). Matched laps take 7.34 s in R and 7.4 s in UR, slower laps 20 s.

In [ ]:
def running_lap_times(c):
    """Code variant (GetBehaviorStats_withGoodrunning.py, good_lap_time): running frames only."""
    return np.array([(c["speed"][s:e] >= K.RUN_SPEED).sum() / K.FS for s, e in K.complete_laps(c)])

for reading in ["R min to max", "R 5th to 95th percentile", "R min to max, running time only (code)"]:
    n_m = n_s = 0
    tm_r, tm_u, ts_u = [], [], []
    med = {"R": [], "matched": [], "slower": []}
    for s in WT:
        lt = RES[s]["beh"]["lap_times"]
        r, u = lt["Rewarded"], lt["Unrewarded"]
        if "code" in reading:
            r = running_lap_times(RES[s]["beh"]["conds"]["Rewarded"])
            u = running_lap_times(RES[s]["beh"]["conds"]["Unrewarded"])
        lo, hi = (np.nanmin(r), np.nanmax(r)) if reading.startswith("R min") else tuple(np.nanpercentile(r, [5, 95]))
        matched = (u >= lo) & (u <= hi)
        n_m += matched.sum()
        n_s += (~matched).sum()
        tm_r += list(r)
        tm_u += list(u[matched])
        ts_u += list(u[~matched])
        ur = lap_r2(s, "Unrewarded")
        med["R"].append(np.nanmean(list(lap_r2(s, "Rewarded").values())))
        med["matched"].append(np.nanmean([ur[k] for k in ur if matched[k]]) if any(matched[k] for k in ur) else np.nan)
        med["slower"].append(np.nanmean([ur[k] for k in ur if not matched[k]]) if any(not matched[k] for k in ur) else np.nan)
    w = stats.wilcoxon(np.array(med["R"]), np.array(med["matched"]))
    print(f"--- {reading}: matched {n_m}, slower {n_s} of {n_m + n_s} UR laps ({100 * n_m / (n_m + n_s):.0f}% matched)")
    print(f"    lap time R {np.nanmean(tm_r):.2f} s, matched UR {np.nanmean(tm_u):.2f} s, slower UR {np.nanmean(ts_u):.2f} s")
    print(f"    median over mice of per-mouse mean R2: R {np.nanmedian(med['R']):.3f}, matched {np.nanmedian(med['matched']):.3f}, slower {np.nanmedian(med['slower']):.3f}")
    print(f"    R vs matched Wilcoxon W={w.statistic:.1f} p={w.pvalue:.4f} (x3 Bonferroni {min(1, 3 * w.pvalue):.4f}), n = {len(WT)} mice")

**Result: partially reproduced.** Decoder R2 falls in UR laps matched to R in lap time as it does in slower laps (median over mice R 0.985, matched 0.763, slower 0.695, all within tolerance; R against matched Wilcoxon p = 0.0059 after Bonferroni). The counts and times are not reproduced: 172 matched and 125 slower laps of 297 (paper 233 and 101), and lap times of 12.23 s in R and 13.38 s in matched UR laps (paper 7.34 and 7.4 s). Counting running time only, as the code does, gives 203 and 94 laps and 8.50 s and 9.16 s, still outside. The timing check in the data check section shows that the 30 Hz clock agrees with the 1.5 s reward pause, so the longer lap times are not a clock error.

## approach-ratio
> Mice slow down approaching the track end; the middle to end velocity ratio is 1.01 in the dark, 1.3 in R and 1.22 in UR, so approach behaviour is an engagement readout.

The dark sessions are not in the dandiset; R and UR are tested. Ratio per lap = mean binned velocity at 100 to 150 cm over 175 to 200 cm. The Methods compute velocity "at each point along the 2 m track", so the main reading takes the lap from its start to the first frame at the track end and leaves out the 1.5 s pause after the reward, during which the mouse stands at 200 cm (first version of this attempt: the whole lap, now a variant). The code's ratio (velocity normalised to its peak, bins 20 to 29 over 28 to 37) is a variant. A synthetic lap at the end of the cell checks the helper.

In [ ]:
appr = {}
for key, label in RATIO_LABELS.items():
    for cond in ["Rewarded", "Unrewarded"]:
        pooled = np.concatenate([RES[s]["beh"][key][cond] for s in WT])
        per_mouse = [np.nanmean(RES[s]["beh"][key][cond]) for s in WT]
        m1, h1 = mean_ci(pooled)
        m2, h2 = mean_ci(per_mouse)
        appr[(key, cond)] = (m1, h1, m2, h2)
        print(f"{label:36s} {cond:10s}: over laps {m1:.3f} +/- {h1:.3f} (n = {np.isfinite(pooled).sum()} laps, {np.isnan(pooled).sum()} undefined); "
              f"over mice {m2:.3f} +/- {h2:.3f} (n = {len(WT)}); median over laps {np.nanmedian(pooled):.3f}; range {np.nanmin(pooled):.2f} to {np.nanmax(pooled):.2f}")
print("laps with an undefined ratio (NaN) per mouse, R / UR:")
print(pd.DataFrame({s: {RATIO_LABELS[k]: f"{np.isnan(RES[s]['beh'][k]['Rewarded']).sum()} / {np.isnan(RES[s]['beh'][k]['Unrewarded']).sum()}" for k in RATIO_LABELS} for s in WT}).to_string())
print("per mouse mean ratio R / UR:")
print(pd.DataFrame({s: {RATIO_LABELS[k]: f"{np.nanmean(RES[s]['beh'][k]['Rewarded']):.2f} / {np.nanmean(RES[s]['beh'][k]['Unrewarded']):.2f}" for k in RATIO_LABELS} for s in WT}).to_string())
# synthetic check of the helper: constant 60 cm/s up to 150 cm, then a linear fall to 30 cm/s at 200 cm, then a 1.5 s pause
x = [0.0]
while x[-1] < 200:
    v = 60.0 if x[-1] < 150 else 60.0 - 30.0 * (x[-1] - 150) / 50
    x.append(min(200.0, x[-1] + v / K.FS))
x = np.array(x + [200.0] * 45)
syn = {"pos": x, "speed": np.r_[np.diff(x) * K.FS, 0.0]}
true = 60.0 / np.mean([60.0 - 30.0 * (p - 150) / 50 for p in np.linspace(175, 200, 101)])
print(f"synthetic lap: true ratio {true:.3f}; helper over the traversal {K.approach_ratio(syn, 0, len(x), True):.3f}, over the whole lap with the pause {K.approach_ratio(syn, 0, len(x)):.3f}")


**Result: not reproduced.** Mice slow down approaching the end in both conditions and more in R than in UR, but the Methods ratio over the traversal is 2.214 +/- 0.092 in R (352 laps) and 1.745 +/- 0.072 in UR (297 laps), against 1.3 +/- 0.02 and 1.22 +/- 0.02 (over mice 2.224 and 1.735). Over the whole lap, which takes in the pause, the ratio is 5.17 and 4.44 with 74 and 76 laps undefined. The code's ratio over the traversal is closest, 1.259 and 1.199, still outside the tolerances of 0.02. On the synthetic lap the helper over the traversal returns the true ratio and the whole-lap reading overstates it. The dark sessions are not in the dandiset.

## engaged-lap-counts
> Approach behaviour stays on most UR laps after licking stops; 170 of 244 RE low laps (70%) are engaged and 74 (30%) disengaged, with 90 RE high laps.

In [ ]:
eng_counts = {}
for ekey in ENG_KEYS:
  print("=== engaged laps by the", RATIO_LABELS[ekey])
  for L in K.LICK_READINGS:
      n_hi = n_eng = n_dis = n_nan = 0
      r_eng, r_dis = [], []
      per = []
      for s in WT:
          b = RES[s]["beh"]
          ls = b["lick_stop"][L]
          e, d = b["engaged_ur"][ekey][ls:], b["disengaged_ur"][ekey][ls:]
          rat = b[ekey]["Unrewarded"][ls:]
          n_hi += ls
          n_eng += e.sum()
          n_dis += d.sum()
          n_nan += (~e & ~d).sum()
          r_eng += list(rat[e])
          r_dis += list(rat[d])
          per.append((s, ls, int(e.sum()), int(d.sum()), round(b["engage_thr"][ekey], 3)))
      eng_counts[(ekey, L)] = (n_eng, n_dis, n_hi, np.nanmean(r_eng), np.nanmean(r_dis))
      tot = n_eng + n_dis
      within = {"engaged": abs(n_eng - 170) <= 17, "disengaged": abs(n_dis - 74) <= 7.4, "RE high": abs(n_hi - 90) <= 9}
      print(f"{L}: RE high {n_hi}, RE low {tot + n_nan}: engaged {n_eng} ({100 * n_eng / tot:.0f}% of classified), disengaged {n_dis}, undefined ratio {n_nan}; "
            f"ratio engaged {np.nanmean(r_eng):.3f} +/- {mean_ci(r_eng)[1]:.3f}, disengaged {np.nanmean(r_dis):.3f} +/- {mean_ci(r_dis)[1]:.3f}; counts within tolerance {within}")
      print("    per mouse (mouse, RE high laps, engaged, disengaged, threshold):", per)


**Result: not reproduced.** With engaged laps defined by the Methods ratio over the traversal (main reading), no RE low lap has an undefined ratio, and 106 of 231 RE low laps (46%) are engaged and 125 disengaged under L1 (100 and 121 under L2, 92 and 119 under L3), against 170 and 74; the RE high count is within tolerance only under L3 (86). The mean ratio of engaged laps is 2.007 and of disengaged laps 1.362, against 1.3 and 1.02. The code's ratio over the traversal, a variant, gives 154 engaged and 77 disengaged laps under L1 (67%), both within tolerance, with ratios of 1.284 and 1.029, but 66 RE high laps against 90 +/- 9. The whole-lap reading of the first version classified 62 of the 231 RE low laps as disengaged only because their ratio was undefined; with those laps left out it leaves 6 disengaged laps.

## supp6b-decoder-engaged-laps
> Decoder R2 is as low in engaged RE low laps (0.65) as in all RE low laps, while disengaged laps reduce it further (0.31).

In [ ]:
for ekey, L in [(K.ENGAGE_MAIN, L) for L in K.LICK_READINGS] + [("ratios_code_trav", "L1")]:
    rows = []
    for s in WT:
        b = RES[s]["beh"]
        ls = b["lick_stop"][L]
        e, dmask = b["engaged_ur"][ekey], b["disengaged_ur"][ekey]
        u = lap_r2(s, "Unrewarded")
        eng = [v for k, v in u.items() if k >= ls and e[k]]
        dis = [v for k, v in u.items() if k >= ls and dmask[k]]
        rows.append({"mouse": s, "R": per_mouse_r2(s, L)["R"], "RE low": per_mouse_r2(s, L)["RE low"],
                     "engaged": np.nanmean(eng) if eng else np.nan, "disengaged": np.nanmean(dis) if dis else np.nan,
                     "n engaged": len(eng), "n disengaged": len(dis)})
    df = pd.DataFrame(rows).set_index("mouse")
    me, he = mean_ci(df["engaged"])
    md, hd = mean_ci(df["disengaged"])
    ok = df[["R", "engaged"]].dropna()
    w = stats.wilcoxon(ok["R"], ok["engaged"]) if len(ok) > 1 else None
    ok2 = df[["engaged", "disengaged"]].dropna()
    print(f"--- {RATIO_LABELS[ekey]}, {L}: engaged {me:.3f} [{me - he:.3f} {me + he:.3f}] (n={df['engaged'].notna().sum()}), disengaged {md:.3f} [{md - hd:.3f} {md + hd:.3f}] (n={df['disengaged'].notna().sum()}); "
          f"R vs engaged Wilcoxon p={w.pvalue if w else np.nan:.4f} (n={len(ok)}); mice with both: {len(ok2)}, engaged > disengaged in {(ok2['engaged'] > ok2['disengaged']).sum()}")
    print(df.round(3).to_string())

**Result: reproduced.** With the main engagement split, under L1, engaged RE low laps decode at R2 0.759 [0.568 0.950] (paper 0.65 +/- 0.16) and disengaged laps at 0.424 [0.214 0.634] (paper 0.31 +/- 0.25), both within tolerance; engaged laps are below R (Wilcoxon p = 0.0020, 10 mice) and disengaged laps are below engaged in 9 of the 10 mice. Under L2 and L3 the means are 0.696 and 0.658 for engaged and 0.448 and 0.444 for disengaged laps, also within tolerance (R against engaged p = 0.0039, 9 mice). With the code's ratio (variant, L1) they are 0.626 and 0.473, engaged below R at p = 0.0020 and above disengaged in 7 of 10 mice. In some mice the engaged mean rests on 1 to 5 laps (NR21, NR32, CFC4, NR34, CFC16 under L1).

## disengaged-laps-uniform
> Disengaged laps are spread evenly through RE low; their probability in the first half of RE low is similar to the second half (P = 0.14) and their distribution does not differ from uniform (P = 0.7).

In [ ]:
for ekey, L in [(K.ENGAGE_MAIN, L) for L in K.LICK_READINGS] + [("ratios_code_trav", "L1")]:
    f1, f2, posn = [], [], []
    for s in WT:
        b = RES[s]["beh"]
        ls = b["lick_stop"][L]
        cl = (b["engaged_ur"][ekey] | b["disengaged_ur"][ekey])[ls:]
        dis = b["disengaged_ur"][ekey][ls:][cl]  # laps with an undefined ratio left out
        n = len(dis)
        if n < 2:
            continue
        h = n // 2
        f1.append(dis[:h].mean())
        f2.append(dis[h:].mean())
        idx = np.where(dis)[0]
        posn += list((idx + 0.5) / n)
    f1, f2 = np.array(f1), np.array(f2)
    sw = stats.shapiro(f1 - f2).pvalue
    t = stats.ttest_rel(f1, f2)
    w = stats.wilcoxon(f1, f2)
    ks = stats.kstest(posn, "uniform")
    print(f"{RATIO_LABELS[ekey]}, {L}: fraction disengaged first half {f1.mean():.3f}, second half {f2.mean():.3f} (n = {len(f1)} mice); Shapiro-Wilk p={sw:.3f}; "
          f"paired t p={t.pvalue:.3f}; Wilcoxon p={w.pvalue:.3f}; KS against uniform on {len(posn)} laps D={ks.statistic:.3f} p={ks.pvalue:.3f}")

**Result: reproduced.** With the main engagement split, the fraction of disengaged laps is 0.383 in the first and 0.541 in the second half of RE low under L1 (paired t p = 0.094; Wilcoxon p = 0.094; 10 mice), and the positions of the 125 disengaged laps do not differ from uniform (KS p = 0.853); L2 and L3 give p from 0.078 to 0.975. Both tests give p >= 0.05 under every reading, as the criterion asks, though the halves differ by 0.16 and 0.17 under L1 and L2. The code's ratio (variant, L1) gives 0.284 and 0.324 (paired t p = 0.639, KS p = 0.541).

## Place fields: the mice of Figures 2 and 3

Figures 2 and 3 use the R-UR-RR mice with more than 12 engaged RE low laps (Methods, Engagement with VR, approach behavior). Place fields are defined separately in R (all laps), in the engaged RE low laps and in RR (all laps), by the criteria in the helper's docstring. With the main engagement split (Methods ratio over the traversal) 2 of the 5 R-UR-RR mice qualify under every lick stop reading (NR6 and NR23), against 3 of 6 in the paper; with the code's ratio (reading "L1c", a variant: lick stop L1, engaged laps by the code's ratio) 4 qualify (NR6, NR21, NR23, CFC4). Every Figure 2 and 3 analysis below is shown under L1 to L3 and L1c; the outcomes are decided on L1 to L3.

In [ ]:
# Readings of the lap sets: L1 to L3 with engaged laps by the main ratio; "L1c" (variant) = lick stop L1
# with engaged laps by the code's ratio over the traversal
READS = K.LICK_READINGS + ["L1c"]

def lsr(L):
    return "L1" if L == "L1c" else L

def eng_key(L):
    return "ratios_code_trav" if L == "L1c" else K.ENGAGE_MAIN

def low_name(L):
    return "RElow_engc_L1" if L == "L1c" else f"RElow_eng_{L}"

def eng_laps(s, L):
    b = RES[s]["beh"]
    e = b["engaged_ur"][eng_key(L)]
    return [i for i in range(b["lick_stop"][lsr(L)], len(e)) if e[i]]

FIG23 = {}
for L in READS:
    counts = {s: len(eng_laps(s, L)) for s in K.WT_FULL}
    FIG23[L] = [s for s in K.WT_FULL if counts[s] > 12]
    print(f"{L}: engaged RE low laps per R-UR-RR mouse {counts}; mice with more than 12: {FIG23[L]}")
print("place fields per mouse and lap set:")
print(pd.DataFrame({s: {k: (len(v["fields"]) if v else None) for k, v in RES[s]["fields"].items()} for s in WT}).to_string())

def fields_df(s, name):
    f = RES[s]["fields"].get(name)
    if not f:
        return pd.DataFrame(columns=["cell", "start", "end", "reliability", "out_in", "width", "intensity", "com", "frac_laps"])
    d = pd.DataFrame(f["fields"])
    d["mouse"] = s
    return d

def cell_com(df):
    """COM of each cell's strongest field."""
    return df.sort_values("intensity").groupby("cell")["com"].last()

def mean_map(s, cond, cells, laps=None):
    m = RES[s]["maps"][cond][cells]
    if laps is not None:
        m = m[:, laps]
    return m.mean(axis=1)

def rowcorr(a, b):
    a = a - a.mean(1, keepdims=True)
    b = b - b.mean(1, keepdims=True)
    den = np.sqrt((a ** 2).sum(1) * (b ** 2).sum(1))
    with np.errstate(invalid="ignore", divide="ignore"):
        return (a * b).sum(1) / den

def group_of(com):
    return np.where(com < 50, "start", np.where(com < 150, "middle", "end"))

## fig2c-spatial-correlation-drop
> Place fields defined in R partially remap in RE low (engaged laps), with a lower spatial correlation between R and RE low than within R, and remap less in RR.

Reading S1 (within-R from control mice) needs the control mice, which the dandiset does not hold; reading S2 (first against second half of the R laps of the same mice) is run. The criterion requires S1 and S2 to agree, so the claim cannot be fully reproduced here whatever S2 gives.

In [ ]:
def spatial_corr(L, mice):
    rows = []
    for s in mice:
        dfR = fields_df(s, "R")
        cells = np.array(sorted(dfR["cell"].unique()))
        nR = RES[s]["maps"]["Rewarded"].shape[1]
        mR = mean_map(s, "Rewarded", cells)
        h1 = mean_map(s, "Rewarded", cells, list(range(nR // 2)))
        h2 = mean_map(s, "Rewarded", cells, list(range(nR // 2, nR)))
        b = RES[s]["beh"]
        low = eng_laps(s, L)
        mlow = mean_map(s, "Unrewarded", cells, low)
        d = pd.DataFrame({"mouse": s, "cell": cells, "com": cell_com(dfR).reindex(cells).values,
                          "within R": rowcorr(h1, h2), "R vs RE low": rowcorr(mR, mlow)})
        if "ReRewarded" in RES[s]["maps"]:
            d["R vs RR"] = rowcorr(mR, mean_map(s, "ReRewarded", cells))
        rows.append(d)
    return pd.concat(rows, ignore_index=True)

fig2c = {}
for L in READS:
    sc = spatial_corr(L, FIG23[L])
    fig2c[L] = sc
    ok = sc[["within R", "R vs RE low", "R vs RR"]].dropna()
    t1 = stats.ttest_rel(ok["R vs RE low"], ok["within R"])
    t2 = stats.ttest_rel(ok["R vs RR"], ok["R vs RE low"])
    t3 = stats.ttest_rel(ok["R vs RR"], ok["within R"])
    print(f"--- {L}, mice {FIG23[L]}: {len(sc)} R place cells, {len(ok)} with all three correlations defined ({len(sc) - len(ok)} dropped: no activity in a lap set)")
    print(f"    mean within R {ok['within R'].mean():.3f}, R vs RE low {ok['R vs RE low'].mean():.3f}, R vs RR {ok['R vs RR'].mean():.3f}")
    print(f"    RE low vs within R t={t1.statistic:.2f} p={t1.pvalue:.2e}; RR vs RE low t={t2.statistic:.2f} p={t2.pvalue:.2e}; RR vs within R t={t3.statistic:.2f} p={t3.pvalue:.2e}")
    print("    per mouse means:\n", ok.join(sc["mouse"]).groupby("mouse").mean(numeric_only=True).round(3).to_string())

**Result: partially reproduced.** Under reading S2, for the 238 R place cells of the 2 qualifying mice with correlations in every lap set (L1), the correlation of R with RE low (0.503) is below the within-R correlation (0.799; paired t p = 1.2e-28), and R with RR (0.579) lies between them, above RE low (p = 8.5e-04); L2 and L3 give the same, and so does the L1c variant with 4 mice (0.556 against 0.857, RR 0.678). The criterion requires readings S1 and S2 to agree; S1 needs the control mice, which are not in the dandiset, so only part of the test can be run and the claim is partially reproduced.

## fig2d-remapping-all-locations
> The drop in spatial correlation in RE low occurs at all track locations, not only near the reward site.

In [ ]:
fig2d = {}
for L in READS:
    sc = fig2c[L].dropna(subset=["within R", "R vs RE low"])
    res_g = {}
    for g in ["start", "middle", "end"]:
        d = sc[group_of(sc["com"].values) == g]
        t = stats.ttest_rel(d["R vs RE low"], d["within R"])
        res_g[g] = (len(d), d["within R"].mean(), d["R vs RE low"].mean(), t.statistic, t.pvalue)
        print(f"{L} {g:6s}: n={len(d)} cells, within R {d['within R'].mean():.3f}, R vs RE low {d['R vs RE low'].mean():.3f}, paired t={t.statistic:.2f} p={t.pvalue:.2e}")
    fig2d[L] = res_g
    print(f"{L}: groups with RE low lower at p < 0.01: {sum(v[2] < v[1] and v[4] < 0.01 for v in res_g.values())} of 3")

**Result: partially reproduced.** Under reading S2 the R against RE low correlation is below within-R at the start (0.514 against 0.820, p = 2.5e-07), middle (0.478 against 0.801, p = 3.9e-17) and end (0.540 against 0.777, p = 2.6e-07) under L1, 3 of 3 groups under every lick stop reading and under L1c. Reading S1, which the criterion requires to agree, cannot be run (no control mice), so the claim is partially reproduced.

## fig3a-field-numbers-similar and the parameters of Figure 3b
> The total number of place fields tiling the track is similar in R, RE low (engaged laps) and RR (605, 663 and 674 fields in 3 mice).

In [ ]:
PF = {}
for L in READS:
    PF[L] = {"R": pd.concat([fields_df(s, "R") for s in FIG23[L]], ignore_index=True),
             "RE low": pd.concat([fields_df(s, low_name(L)) for s in FIG23[L]], ignore_index=True),
             "RR": pd.concat([fields_df(s, "RR") for s in FIG23[L]], ignore_index=True)}
    n = {k: len(v) for k, v in PF[L].items()}
    within = {k: abs(n[k] - v) <= tol for k, (v, tol) in {"R": (605, 60), "RE low": (663, 66), "RR": (674, 67)}.items()}
    print(f"{L}, mice {FIG23[L]}: fields R {n['R']}, RE low engaged {n['RE low']}, RR {n['RR']}; within tolerance {within}")
    print("    per mouse:", {k: v.groupby("mouse").size().to_dict() for k, v in PF[L].items()})
    print("    place cells (cells with a field) per condition:", {k: v.groupby("mouse")["cell"].nunique().sum() for k, v in PF[L].items()})

**Result: not reproduced.** In the 2 qualifying mice, R has 245 fields, the engaged RE low laps 151 (L1; 149 and 146 under L2 and L3) and RR 254, so the RE low count is 38% below R and the numbers are not similar; all three are far from 605, 663 and 674, which the paper counts in 3 mice. With 4 mice (L1c) the counts are 747, 332 and 736: RR is within tolerance and RE low again far below R.

## fig3b-reliability, fig3b-out-in-ratio, fig3b-width-intensity
> Place field trial to trial reliability falls from 0.47 in R to 0.32 in RE low (engaged laps) and returns towards R in RR (0.41). The out/in place field firing ratio rises from 0.10 in R to 0.16 in RE low and returns towards R in RR (0.12). Place fields widen from 53.34 cm in R to 60.56 cm in RE low (54.08 in RR), and firing intensity falls slightly from 0.36 to 0.31 and stays low in RR (0.29).

In [ ]:
EXP3B = {"reliability": {"R": (0.47, 0.10), "RE low": (0.32, 0.17), "RR": (0.41, 0.10)},
         "out_in": {"R": (0.10, 0.03), "RE low": (0.16, 0.04), "RR": (0.12, 0.04)},
         "width": {"R": (53.34, 9.47), "RE low": (60.56, 12.5), "RR": (54.08, 7.23)},
         "intensity": {"R": (0.36, 0.05), "RE low": (0.31, 0.09), "RR": (0.29, 0.05)}}
fig3b = {}
for L in READS:
    print(f"--- {L}")
    for p, ex in EXP3B.items():
        for reading in ["per-mouse means", "pooled fields"]:
            vals = {}
            for c in ["R", "RE low", "RR"]:
                d = PF[L][c]
                if reading == "per-mouse means":
                    pm = d.groupby("mouse")[p].mean()
                    vals[c] = mean_ci(pm)
                else:
                    vals[c] = mean_ci(d[p])
            within = {c: abs(vals[c][0] - ex[c][0]) <= ex[c][1] for c in ex}
            fig3b[(L, p, reading)] = (vals, within)
            print(f"  {p:11s} {reading:16s}: " + "; ".join(f"{c} {m:.3f} +/- {h:.3f}" for c, (m, h) in vals.items()) + f"  within tolerance {within}")
        a, b2 = PF[L]["R"][p].dropna(), PF[L]["RE low"][p].dropna()
        t = stats.ttest_ind(a, b2)
        ks = stats.ks_2samp(a, b2)
        print(f"  {p:11s} R vs RE low pooled fields: unpaired t={t.statistic:.2f} p={t.pvalue:.2e}; KS D={ks.statistic:.3f} p={ks.pvalue:.2e}; range R {a.min():.2f} to {a.max():.2f}, RE low {b2.min():.2f} to {b2.max():.2f}")
print("per mouse (L1):")
print(pd.concat({c: PF["L1"][c].groupby("mouse")[["reliability", "out_in", "width", "intensity"]].mean() for c in PF["L1"]}).round(3).to_string())
# width variant: the extent of the candidate field (bins above 25% of the way from baseline to peak), which does not use the
# "mean above 0" extent of the field-only map
for L in READS:
    print(f"{L} candidate-field width, per-mouse means (cm):", {c: round(((PF[L][c]["end"] - PF[L][c]["start"]) * K.BIN).groupby(PF[L][c]["mouse"]).mean().mean(), 1) for c in PF[L]})


**Result: reproduced.** Over the per-mouse means of the 2 qualifying mice, reliability is 0.545 in R, 0.381 in RE low and 0.495 in RR (L1), within the tolerances of 0.47 +/- 0.10, 0.32 +/- 0.17 and 0.41 +/- 0.10, and RE low is below R (pooled fields, unpaired t p = 2.0e-26; KS p = 2.4e-20); the pooled fields give 0.554, 0.351 and 0.478, and L2 and L3 also hold. With 4 mice (L1c) R (0.570) and RR (0.520) are just outside their tolerances.

**Result: not reproduced.** The out/in ratio rises from R to RE low (pooled fields, unpaired t p = 1.4e-03 under L1), but the per-mouse means are R 0.068, RE low 0.086 and RR 0.075 under L1 (0.087 and 0.092 for RE low under L2 and L3), against 0.10 +/- 0.03, 0.16 +/- 0.04 and 0.12 +/- 0.04: R is just below its band and RE low well below it under every reading, per mouse or pooled. With 4 mice (L1c) R (0.074) and RR (0.080) are within and RE low (0.105) is not. The first version passed under L2 only, with the RE low laps of the old engagement split.

**Result: not reproduced.** Fields are wider than the paper's: per-mouse means of 70.1 cm in R, 65.6 cm in RE low and 73.1 cm in RR (L1), against 53.34, 60.56 and 54.08 cm; only RE low is within tolerance, and RE low is not wider than R (pooled p = 0.24). Firing intensity is within tolerance in R (0.380) and RE low (0.311) and falls from R to RE low (pooled p = 8.7e-04), but RR (0.393) is above 0.29 +/- 0.05. The candidate regions that the field search cuts (bins above 25% of the way from baseline to peak) are 31 to 35 cm wide on average; the width rule of the Methods, from where the field-only mean map rises above 0 to where it returns to 0, roughly doubles that.

## fig3c-degradation-all-locations
> The degradation of place fields in RE low (reliability and out/in ratio) is seen at all track locations.

In [ ]:
for L in READS:
    n_ok = {}
    for p, direction in [("reliability", -1), ("out_in", 1)]:
        k = 0
        for g in ["start", "middle", "end"]:
            a = PF[L]["R"][group_of(PF[L]["R"]["com"].values) == g][p].dropna()
            b2 = PF[L]["RE low"][group_of(PF[L]["RE low"]["com"].values) == g][p].dropna()
            ks = stats.ks_2samp(a, b2)
            ok = (np.sign(b2.mean() - a.mean()) == direction) and ks.pvalue < 0.01
            k += ok
            print(f"{L} {p:11s} {g:6s}: R {a.mean():.3f} (n={len(a)}), RE low {b2.mean():.3f} (n={len(b2)}), KS D={ks.statistic:.3f} p={ks.pvalue:.2e}, holds {ok}")
        n_ok[p] = k
    print(f"{L}: groups that hold, reliability {n_ok['reliability']} of 3, out/in {n_ok['out_in']} of 3")

**Result: partially reproduced.** Under L1, reliability is lower in RE low at the start, middle and end (KS p 4.7e-07 to 2.8e-09), and the out/in ratio is higher in the middle (p = 2.7e-03) but not at the start (p = 0.22) or the end (p = 0.013): 3 groups for one measure, which the criterion counts as partial; L2 and L3 give the same, and L1c gives 3 and 2 groups.

## supp11-all-mice-parameters
> With all mice and both engaged and disengaged RE low laps, place field medians show the same degradation and return.

In [ ]:
EXP11 = {"reliability": (0.48, 0.26, 0.44), "out_in": (0.10, 0.19, 0.13), "width": (50.21, 54.96, 45.71), "intensity": (0.34, 0.26, 0.28)}
supp11 = {}
for L in K.LICK_READINGS:
    pf = {"R": pd.concat([fields_df(s, "R") for s in WT]), "RE low": pd.concat([fields_df(s, f"RElow_all_{L}") for s in WT]),
          "RR": pd.concat([fields_df(s, "RR") for s in K.WT_FULL])}
    for reading in ["(a) median of per-mouse means", "(b) median of pooled fields"]:
        n_in = 0
        dirs = []
        line = []
        for p, ex in EXP11.items():
            v = []
            for c in ["R", "RE low", "RR"]:
                if reading.startswith("(a)"):
                    v.append(pf[c].groupby("mouse")[p].mean().median())
                else:
                    v.append(pf[c][p].median())
            n_in += sum(abs(x - e) <= 0.1 * e for x, e in zip(v, ex))
            dirs.append((v[1] < v[0]) if p in ("reliability", "intensity") else (v[1] > v[0]))
            line.append(f"{p} " + "/".join(f"{x:.3f}" for x in v))
        supp11[(L, reading)] = (n_in, all(dirs))
        print(f"{L} {reading}: " + "; ".join(line) + f" | within 10%: {n_in} of 12; directions hold: {all(dirs)} {dirs}")
    print(f"    fields: R {len(pf['R'])} (10 mice), RE low {len(pf['RE low'])}, RR {len(pf['RR'])} (5 mice)")

**Result: not reproduced.** With all 10 mice and all RE low laps, every direction holds (reliability and intensity lower and out/in ratio and width higher in RE low), but only 3 of the 12 medians are within 10% under reading (b) and none under reading (a); the criterion asks for 10 (7 for partial). Widths are 75 to 81 cm against 46 to 55 cm. This claim does not use the engagement split.

## fig2e-place-field-fates
> Of 605 R place fields (3 mice), 169 (27.9%) stay stable through R, RE low and RR; in RE low 222 (36.7%) stay stable, 249 (41.2%) disappear and 134 (22.1%) remap, and 296 of the 663 RE low fields (44.6%) are new. In RR 311 R fields (51.4%) are stable and 89 (14.7%) remapped, 126 of the 249 lost fields (50.6%) reappear, 64 of the 296 new RE low fields (21.6%) stay and 204 (68.9%) disappear, and 140 of 674 RR fields (20.8%) are new.

In [ ]:
def is_stable(f_early, f_late, rule):
    if rule == "F1":
        return f_late["start"] < f_early["end"] and f_early["start"] < f_late["end"]
    if rule == "F2":
        return f_early["start"] * K.BIN <= f_late["com"] < f_early["end"] * K.BIN
    return group_of(np.array([f_early["com"]]))[0] == group_of(np.array([f_late["com"]]))[0]

def match(f, later_cell_fields):
    if later_cell_fields is None or len(later_cell_fields) == 0:
        return None
    i = np.argmin(np.abs(later_cell_fields["com"].values - f["com"]))
    return later_cell_fields.iloc[i]

def fates(L, rule):
    out = {k: 0 for k in ["R", "RE low", "RR", "stable through", "stable RE low", "lost RE low", "remap RE low", "new RE low",
                          "stable RR", "remap RR", "lost RR", "lost reappear RR", "new low stable RR", "new low lost RR", "new RR (vs RE low)", "new RR (vs R and RE low)",
                          "lost end", "lost end reappear"]}
    newlow_bins = np.zeros(4, int)
    newrr_bins = np.zeros(4, int)
    for s in FIG23[L]:
        R, Lo, RR = fields_df(s, "R"), fields_df(s, low_name(L)), fields_df(s, "RR")
        gR, gL, gRR = R.groupby("cell"), Lo.groupby("cell"), RR.groupby("cell")
        cR, cL, cRR = set(R["cell"]), set(Lo["cell"]), set(RR["cell"])
        out["R"] += len(R); out["RE low"] += len(Lo); out["RR"] += len(RR)
        for _, f in R.iterrows():
            ml = match(f, gL.get_group(f["cell"]) if f["cell"] in cL else None)
            mr = match(f, gRR.get_group(f["cell"]) if f["cell"] in cRR else None)
            st_low = ml is not None and is_stable(f, ml, rule)
            st_rr = mr is not None and is_stable(f, mr, rule)
            if ml is None:
                out["lost RE low"] += 1
                if f["com"] >= 150:
                    out["lost end"] += 1
                if st_rr:
                    out["lost reappear RR"] += 1
                    if f["com"] >= 150:
                        out["lost end reappear"] += 1
            elif st_low:
                out["stable RE low"] += 1
            else:
                out["remap RE low"] += 1
            if mr is None:
                out["lost RR"] += 1
            elif st_rr:
                out["stable RR"] += 1
            else:
                out["remap RR"] += 1
            if st_low and st_rr:
                out["stable through"] += 1
        for _, f in Lo.iterrows():
            if f["cell"] not in cR:
                out["new RE low"] += 1
                newlow_bins[min(int(f["com"] // 50), 3)] += 1
                mr = match(f, gRR.get_group(f["cell"]) if f["cell"] in cRR else None)
                if mr is None:
                    out["new low lost RR"] += 1
                elif is_stable(f, mr, rule):
                    out["new low stable RR"] += 1
        for _, f in RR.iterrows():
            if f["cell"] not in cL:
                out["new RR (vs RE low)"] += 1
                if f["cell"] not in cR:
                    out["new RR (vs R and RE low)"] += 1
                    newrr_bins[min(int(f["com"] // 50), 3)] += 1
    return out, newlow_bins, newrr_bins

FATES = {}
for L in READS:
    for rule in ["F1", "F2", "F3"]:
        o, nlb, nrb = fates(L, rule)
        FATES[(L, rule)] = (o, nlb, nrb)
        pct = lambda a, b: 100 * a / b if b else np.nan
        vals = {"stable through": pct(o["stable through"], o["R"]), "stable RE low": pct(o["stable RE low"], o["R"]),
                "lost RE low": pct(o["lost RE low"], o["R"]), "remap RE low": pct(o["remap RE low"], o["R"]),
                "new RE low": pct(o["new RE low"], o["RE low"]), "stable RR": pct(o["stable RR"], o["R"]),
                "remap RR": pct(o["remap RR"], o["R"]), "lost reappear": pct(o["lost reappear RR"], o["lost RE low"]),
                "new low stable RR": pct(o["new low stable RR"], o["new RE low"]), "new low lost RR": pct(o["new low lost RR"], o["new RE low"]),
                "new RR (vs R and RE low)": pct(o["new RR (vs R and RE low)"], o["RR"]), "new RR (vs RE low)": pct(o["new RR (vs RE low)"], o["RR"])}
        FATES[(L, rule)] = (o, nlb, nrb, vals)
        print(f"{L} {rule}: fields R {o['R']}, RE low {o['RE low']}, RR {o['RR']} | " + "; ".join(f"{k} {v:.1f}%" for k, v in vals.items()))
EXP2E = {"stable through": (27.9, 2.8), "stable RE low": (36.7, 3.7), "lost RE low": (41.2, 4.1), "remap RE low": (22.1, 2.2), "new RE low": (44.6, 4.5),
         "stable RR": (51.4, 5.1), "remap RR": (14.7, 1.5), "lost reappear": (50.6, 5.1), "new low stable RR": (21.6, 2.2), "new low lost RR": (68.9, 6.9),
         "new RR (vs R and RE low)": (20.8, 2.1)}
for k, (o, nlb, nrb, vals) in FATES.items():
    w = [n for n, (v, tol) in EXP2E.items() if abs(vals[n] - v) <= tol]
    print(k, f"within tolerance: {len(w)} of {len(EXP2E)}:", w)

**Result: not reproduced.** In the 2 qualifying mice, 75.9% of the 245 R fields disappear in RE low under L1 (paper 41.2%) and 60.3% of the 151 RE low fields are new (paper 44.6%); these do not depend on the stable/remapped definitions. Under F1 to F3, 0 or 1 of the 11 percentages is within tolerance under any lick stop reading, and 0 under L1c (4 mice, 77.5% lost).

## reward-zone-field-turnover
> Of the R place fields that disappear in RE low, 41.8% (104 of 249) are at the track end (150 to 200 cm); new RE low fields are spread evenly (26, 28, 21 and 25% per 50 cm). In RR only 61 of the 104 lost end fields return (58.6%), and new RR fields cluster at the end (31, 24, 18 and 67 of 140, 48% at 150 to 200 cm).

In [ ]:
for k, (o, nlb, nrb, vals) in FATES.items():
    print(f"{k}: lost {o['lost RE low']}, of them at 150 to 200 cm {o['lost end']} ({100 * o['lost end'] / max(1, o['lost RE low']):.1f}%); "
          f"new RE low per 50 cm {nlb.tolist()} ({np.round(100 * nlb / max(1, nlb.sum())).astype(int).tolist()}%); "
          f"lost end fields reappearing in RR {o['lost end reappear']}; new RR per 50 cm {nrb.tolist()} ({np.round(100 * nrb / max(1, nrb.sum())).astype(int).tolist()}%)")

**Result: not reproduced.** Of the 186 R fields lost in RE low under L1, 50 (26.9%) are at 150 to 200 cm, against 104 of 249 (41.8%); new RE low fields are 23, 18, 33 and 26% per 50 cm (paper 26, 28, 21 and 25%) and number 91, not 296; 15 to 21 lost end fields reappear in RR (paper 61), and new RR fields are 29, 18, 17 and 27 per 50 cm (paper 31, 24, 18 and 67). Most counts are outside tolerance under every reading: the lost end fields, the four bins of new RE low fields, the lost end fields that reappear in RR, and new RR fields at 50 to 100 cm (16 to 18 against 24 +/- 2.4) and at 150 to 200 cm (26 to 28 against 67). Two are inside: new RR fields at 0 to 50 cm (29, 31 and 29 under L1 to L3, against 31 +/- 3.1) and at 100 to 150 cm under L1 and L2 (17, against 18 +/- 1.8; 22 under L3); with 4 mice (L1c) 200 of 579 lost fields (34.5%) are at the end.

## fig3d-reward-overrepresentation
> Place fields over-represent the reward site at the track end in R; the over-representation disappears in RE low and reappears in RR. Gaussian fits to field COMs are 114 +/- 55 cm in R, 102 +/- 54 in UR and 112 +/- 58 in RR.

In [ ]:
def end_minus_middle(df):
    com = df["com"].dropna().values
    if len(com) == 0:
        return np.nan
    h = np.histogram(com, bins=[0, 50, 100, 150, 200])[0] / len(com) * 100
    return h[3] - h[1:3].mean()

for L in READS:
    em = {c: [end_minus_middle(PF[L][c][PF[L][c]["mouse"] == s]) for s in FIG23[L]] for c in PF[L]}
    gauss = {c: stats.norm.fit(PF[L][c]["com"].dropna()) for c in PF[L]}
    print(f"--- {L}, mice {FIG23[L]}")
    print("   end minus middle (% of fields) per mouse:", {c: np.round(v, 1).tolist() for c, v in em.items()})
    print("   Gaussian fit mean +/- SD (cm):", {c: (round(m, 1), round(sd, 1)) for c, (m, sd) in gauss.items()})
    for c, v in em.items():
        t = stats.ttest_1samp(v, 0)
        print(f"   {c}: mean {np.mean(v):.1f}, t-test against 0 p={t.pvalue:.3f}")
    print("   R > 0 in all mice:", all(np.array(em["R"]) > 0), "| RR > 0 in all mice:", all(np.array(em["RR"]) > 0),
          "| RE low below R in all mice:", all(np.array(em["RE low"]) < np.array(em["R"])), "| RE low mean <= 0:", np.mean(em["RE low"]) <= 0)

**Result: not reproduced.** In the 2 qualifying mice the end minus middle excess is 6.8 and -4.2 points in R, so R does not over-represent the reward site in every mouse; in RE low it is -14.6 and 5.5 (mean -4.6, not above 0) and it is not below R in NR23; in RR it is -5.7 and 9.5. The Gaussian fits are within tolerance (R 106.4 +/- 58.6, RE low 108.4 +/- 54.7, RR 103.8 +/- 57.9 cm under L1). With 4 mice (L1c), R is above 0 in 3 of 4 mice (6.8, 13.1, -4.2, 17.7), RE low is below R in 3 of 4 mice with a mean of -7.4, and RR is above 0 in 3 of 4.

## fig2a-pv-cluster-transition
> Lap by lap population vector correlations of R place cells show a transition around the lick stop (RE high to RE low) and again after reward returns; the probability of laps belonging to the R cluster dips from RE high to RE low, not from R to RE high, and rises to some extent in RR (fully only in mouse 1).

Per lap, the population vector is each R place cell's mean dF/F in each of the 40 bins. Laps: R, RE high, RE low engaged, RR. k-means with 3 clusters, run 1000 times from random starts (seeded); the R cluster is the one holding most R laps in each run.

In [ ]:
from sklearn.cluster import KMeans
rng_pv = np.random.default_rng(2)
fig2a = {}


def pv_clusters(s, L, n_r=None, tag=""):
    """R-cluster probability per lap set; n_r keeps only the first n_r R laps (variant)."""
    cells = np.array(sorted(fields_df(s, "R")["cell"].unique()))
    b = RES[s]["beh"]
    ls = b["lick_stop"][lsr(L)]
    nR = RES[s]["maps"]["Rewarded"].shape[1] if n_r is None else n_r
    parts = [("R", "Rewarded", list(range(nR))),
             ("RE high", "Unrewarded", list(range(ls))),
             ("RE low", "Unrewarded", eng_laps(s, L)),
             ("RR", "ReRewarded", list(range(RES[s]["maps"]["ReRewarded"].shape[1])))]
    X, lab = [], []
    for name, cond, laps in parts:
        m = RES[s]["maps"][cond][cells][:, laps]  # cells x laps x 40
        X.append(m.transpose(1, 0, 2).reshape(len(laps), -1))
        lab += [name] * len(laps)
    X = np.vstack(X)
    lab = np.array(lab)
    pvc = np.corrcoef(X)
    prob = np.zeros(len(X))
    n_runs = 1000
    for i in range(n_runs):
        km = KMeans(3, n_init=1, random_state=int(rng_pv.integers(1 << 31))).fit(X)
        rc = np.bincount(km.labels_[lab == "R"], minlength=3).argmax()
        prob += km.labels_ == rc
    prob /= n_runs
    mp = {n: prob[lab == n].mean() for n in ["R", "RE high", "RE low", "RR"]}
    ok = abs(mp["RE high"] - mp["R"]) <= 0.2 and mp["RE low"] <= mp["RE high"] - 0.3
    mean_pvc = {n: pvc[np.ix_(lab == n, lab == "R")].mean() for n in ["R", "RE high", "RE low", "RR"]}
    print(f"{L}{tag} {s}: {len(cells)} R place cells, laps {dict(zip(*np.unique(lab, return_counts=True)))}; mean R-cluster probability "
          + ", ".join(f"{n} {v:.2f}" for n, v in mp.items()) + f"; holds {ok}")
    print("    mean PV correlation with R laps: " + ", ".join(f"{n} {v:.3f}" for n, v in mean_pvc.items()))
    print("    lap-wise probability:", " ".join(f"{x:.2f}" for x in prob))
    return ok

for L in READS:
    holds = 0
    for s in FIG23[L]:
        ok = pv_clusters(s, L)
        holds += ok
    fig2a[L] = holds
    print(f"{L}: holds in {holds} of {len(FIG23[L])} mice")

# Variant: the paper's code keeps only the first 11100 R frames of NR6 (task_framestokeep), here the
# R laps that end within them; the R place cells stay those found on all R laps
n_nr6 = code_kept_laps("NR6", 0, RES["NR6"]["beh"]["conds"]["Rewarded"])
print(f"NR6 R laps kept by the code's frame cut: {n_nr6} of {RES['NR6']['maps']['Rewarded'].shape[1]}")
for L in READS:
    if "NR6" in FIG23[L]:
        pv_clusters("NR6", L, n_r=n_nr6, tag=" (NR6 R cut)")


**Result: not reproduced.** In the 2 qualifying mice the rule holds in neither under any lick stop reading: in NR6 the R-cluster probability falls already at the first UR lap (R 0.87, RE high 0.10, RE low 0.05), and in NR23, with 2 to 4 RE high laps, RE high is already low (0.49 against R 0.82 under L1, RE low 0.23). With 4 mice (L1c, a variant) it holds in 2 of 4 (NR21: R 0.84, RE high 0.82, RE low 0.15; CFC4: 0.89, 0.72, 0.14), the paper's pattern. RR probabilities are low (0.02 to 0.33). Keeping only the first 24 of NR6's 43 R laps, as the paper's code does (task_framestokeep, 11100 frames; R place cells unchanged), NR6 still fails under every reading (R 0.96, RE high 0.08 to 0.10, RE low 0.09 to 0.12); NR6's Figure 2 and 3 field values with fields found on those 24 laps were not computed.

## supp13-novel-environment
> In a novel, never reinforced environment, decoding and place field reliability and out/in ratio are poor in early laps and improve over the session as mice develop reward expectation, becoming comparable to R and RR; RE low reliability and out/in ratio match the early novel laps.

Five of the six mice with a novel session are in the dandiset. Early = first 12 novel laps, late = last 12. Decoders are trained and tested within each condition (sliding 20% test block), each condition down-sampled to the fewest laps; for early and late R2 the novel session is decoded on all its laps. Place field parameters in a block are read two ways: fields defined in the block, or fields defined over the whole novel session and measured on the block's laps.

In [ ]:
def block_params(s, name_all, laps):
    """Parameters of the session's novel fields measured on a block of laps."""
    df = fields_df(s, name_all)
    M = RES[s]["maps"]["Novel"][:, laps]
    out = []
    for _, f in df.iterrows():
        lm = M[int(f["cell"])]
        p = K.field_parameters(lm, lm > 0, int(f["start"]), int(f["end"]))
        out.append(p)
    return pd.DataFrame(out)

def prelicks_per_lap(c, zone):
    """Licks in the reward zone before the reward is delivered, per lap (pre-licking)."""
    out = []
    for a, e in K.complete_laps(c):
        rw = c["rewards"][(c["rewards"] >= a) & (c["rewards"] < e)]
        stop = rw[0] if len(rw) else e
        lk = c["licks"][(c["licks"] >= a) & (c["licks"] < stop)]
        out.append(int((c["pos"][lk] >= zone).sum()))
    return np.array(out)

rows = []
for s in K.WT_FULL:
    b = RES[s]["beh"]
    cN = b["conds"]["Novel"]
    nN = len(K.complete_laps(cN))
    early, late = list(range(min(12, nN))), list(range(max(0, nN - 12), nN))
    zl = prelicks_per_lap(cN, b["zone"])
    cv = RES[s]["dec_cv_novel_all"]
    r = {"mouse": s, "novel laps": nN,
         "pre-licking early": np.mean(zl[early]), "pre-licking late": np.mean(zl[late]),
         "R2 early": np.nanmean(cv[early]), "R2 late": np.nanmean(cv[late]),
         "R2 R (cv)": np.nanmean(RES[s]["dec_cv"]["Rewarded"]), "R2 RR (cv)": np.nanmean(RES[s]["dec_cv"]["ReRewarded"]),
         "R2 N (cv)": np.nanmean(RES[s]["dec_cv"]["Novel"])}
    for p in ["reliability", "out_in"]:
        r[f"{p} early (block fields)"] = fields_df(s, "N_early")[p].mean()
        r[f"{p} late (block fields)"] = fields_df(s, "N_late")[p].mean()
        r[f"{p} early (session fields)"] = block_params(s, "N", early)[p].mean()
        r[f"{p} late (session fields)"] = block_params(s, "N", late)[p].mean()
        r[f"{p} R"] = fields_df(s, "R")[p].mean()
        r[f"{p} RR"] = fields_df(s, "RR")[p].mean()
        r[f"{p} RE low (L1, all laps)"] = fields_df(s, "RElow_all_L1")[p].mean()
    rows.append(r)
nov = pd.DataFrame(rows).set_index("mouse")
print(nov.T.round(3).to_string())
tests = {}
for label, a, b2, better in [("pre-licking", "pre-licking early", "pre-licking late", 1), ("decoder R2", "R2 early", "R2 late", 1),
                             ("reliability (block fields)", "reliability early (block fields)", "reliability late (block fields)", 1),
                             ("reliability (session fields)", "reliability early (session fields)", "reliability late (session fields)", 1),
                             ("out/in (block fields)", "out_in early (block fields)", "out_in late (block fields)", -1),
                             ("out/in (session fields)", "out_in early (session fields)", "out_in late (session fields)", -1)]:
    d = nov[[a, b2]].dropna()
    t = stats.ttest_rel(d[b2], d[a])
    ok = np.sign(d[b2].mean() - d[a].mean()) == better and t.pvalue < 0.05
    tests[label] = ok
    print(f"{label:30s}: early {d[a].mean():.3f}, late {d[b2].mean():.3f}, paired t={t.statistic:.2f} p={t.pvalue:.4f} (n={len(d)}); improves at p < 0.05: {ok}")
# scale of the session-field out/in ratio on a block of 12 laps: fields with little activity in the block give large ratios
for s in K.WT_FULL:
    nN = len(K.complete_laps(RES[s]["beh"]["conds"]["Novel"]))
    for blk, laps in [("early", list(range(min(12, nN)))), ("late", list(range(max(0, nN - 12), nN)))]:
        bp = block_params(s, "N", laps)["out_in"]
        print(f"{s} session fields, {blk} block: out/in median {bp.median():.3f}, mean {bp.mean():.3f}, undefined (no activity in field) {bp.isna().sum()} of {len(bp)}, above 1: {(bp > 1).sum()}")


**Result: partially reproduced.** In the 5 novel mice, late laps decode better than early laps (R2 0.821 to 0.919, p = 0.0093) and have higher reliability (block fields 0.320 to 0.392, p = 0.038; session fields 0.230 to 0.434, p = 0.0036). Pre-licking in the reward zone before the reward (p = 0.39) and the out/in ratio (p = 0.34 and 0.096) do not change significantly: 2 of the 4 tests hold. The session-field out/in means (0.550 early) are pulled up by fields with almost no activity in a 12-lap block (up to 20 fields with a ratio above 1 per mouse); their medians are 0.121 to 0.158 early and 0.040 to 0.083 late.

## DREADD mice

Four DAT-Cre hM4D(Gi) mice (sub-009, 011, 012, 014); the paper has six (five for saline). Day 1 (R, UR, RR) and days 2 to 4 (before and after saline, CNO and DCZ) are stored back to back in one file per mouse; 012 has no RR.

In [ ]:
PAIRS = {"saline": ("BeforeSaline", "AfterSaline"), "DCZ": ("BeforeDCZ", "AfterDCZ"), "CNO": ("BeforeCNO", "AfterCNO")}
for s in DR:
    b = RES[s]["beh"]
    print(s, {n: len(K.complete_laps(c)) for n, c in b["conds"].items()}, "lick stop", b["lick_stop"], "engaged RE low (L1)", len(eng_laps(s, "L1")))

## supp16-dreadd-running-speed
> DCZ and CNO slow lap running speed in R (from about 42 to about 26), while saline does not, and approach behaviour and anticipatory licking remain.

In [ ]:
EXP16 = {"BeforeSaline": (42.74, 1.62), "AfterSaline": (42.75, 1.42), "BeforeDCZ": (41.99, 1.74), "AfterDCZ": (25.96, 1.66), "BeforeCNO": (43.18, 1.92), "AfterCNO": (25.71, 1.68)}

def run_speed(c):
    return np.array([np.mean(c["speed"][a:e][c["speed"][a:e] >= K.RUN_SPEED]) for a, e in K.complete_laps(c)])

def prelick_laps(c, zone):
    """Fraction of laps with a lick in the reward zone before the reward."""
    out = []
    for a, e in K.complete_laps(c):
        rw = c["rewards"][(c["rewards"] >= a) & (c["rewards"] < e)]
        stop = rw[0] if len(rw) else e
        lk = c["licks"][(c["licks"] >= a) & (c["licks"] < stop)]
        out.append(bool((c["pos"][lk] >= zone).any()))
    return np.mean(out)

for measure, fn in [("lap speed 200 cm / lap time", lambda c: K.lap_speeds(c)), ("mean running speed", run_speed)]:
    print(f"--- {measure} (cm/s)")
    for reading in ["laps pooled", "per-mouse means"]:
        vals = {}
        for n in EXP16:
            per = [fn(RES[s]["beh"]["conds"][n]) for s in DR]
            vals[n] = mean_ci(np.concatenate(per)) if reading == "laps pooled" else mean_ci([np.nanmean(p) for p in per])
        within = {n: abs(vals[n][0] - v) <= tol for n, (v, tol) in EXP16.items()}
        print(f"  {reading}: " + "; ".join(f"{n} {m:.2f} [{m - h:.2f} {m + h:.2f}]" for n, (m, h) in vals.items()) + f" | within {sum(within.values())} of 6")
    for drug, (bn, an) in PAIRS.items():
        before = np.concatenate([fn(RES[s]["beh"]["conds"][bn]) for s in DR])
        after = np.concatenate([fn(RES[s]["beh"]["conds"][an]) for s in DR])
        t = stats.ttest_ind(after[np.isfinite(after)], before[np.isfinite(before)])
        print(f"  {drug}: after minus before {np.nanmean(after) - np.nanmean(before):.2f}, unpaired t over laps t={t.statistic:.2f} p={t.pvalue:.2e} ({np.isfinite(before).sum()} and {np.isfinite(after).sum()} laps)")
for s in DR:
    print(s, "lap speed per condition:", {n: round(np.nanmean(K.lap_speeds(RES[s]["beh"]["conds"][n])), 1) for n in EXP16},
          "| laps with a pre-reward lick in the zone:", {n: round(prelick_laps(RES[s]["beh"]["conds"][n], RES[s]["beh"]["zone"]), 2) for n in EXP16},
          "| approach ratio (main reading):", {n: round(np.nanmean(RES[s]["beh"][K.ENGAGE_MAIN][n]), 2) for n in EXP16})

**Result: not reproduced.** DCZ and CNO lower lap speed (after minus before -13.70 and -13.05 cm/s, unpaired t p = 7.9e-38 and 2.0e-29) and saline does not (+0.22, p = 0.78), and nearly all laps after DCZ and CNO still have a pre-reward lick in the zone. But none of the six means is within tolerance: 25.9 to 27.6 cm/s before and 13.5 to 14.6 cm/s after, against about 42 and 26, with the 4 of 6 mice in the dandiset. The criterion needs the six means. The timing check shows that the clock agrees with the reward pause, and lap speed is the 200 cm track over the lap time, so it does not depend on the volts-to-cm scale; the slower speeds are in the data or in a lap speed measure the paper does not define.

## fig4f-dreadd-spatial-correlation
> Spatial correlation of place fields before and after DCZ (and CNO) falls at all track locations, as in RE low, while it does not fall after saline.

In [ ]:
def pair_corr(s, first, second, second_laps=None):
    df = fields_df(s, first)
    cells = np.array(sorted(df["cell"].unique()))
    c1 = first if first in RES[s]["maps"] else RES[s]["fields"][first]["cond"]
    laps1 = None if first in RES[s]["maps"] else RES[s]["fields"][first]["laps"]
    m1 = mean_map(s, c1, cells, laps1)
    m2 = mean_map(s, second, cells, second_laps)
    return pd.DataFrame({"mouse": s, "cell": cells, "com": cell_com(df).reindex(cells).values, "r": rowcorr(m1, m2)})

def dreadd_corrs(L, lowset):
    out = {}
    rows = []
    for s in DR:
        b = RES[s]["beh"]
        low = eng_laps(s, L) if lowset == "engaged" else list(range(b["lick_stop"][L], len(b["engaged_ur"][K.ENGAGE_MAIN])))
        rows.append(pair_corr(s, "Rewarded", "Unrewarded", low))
    out["R/RE low"] = pd.concat(rows, ignore_index=True)
    for drug, (bn, an) in PAIRS.items():
        out[drug] = pd.concat([pair_corr(s, bn, an) for s in DR], ignore_index=True)
    return out

EXP4F = {"R/RE low": (928, 93), "saline": (1139, 114), "DCZ": (1629, 163), "CNO": (1498, 150)}
fig4f = {}
for L in K.LICK_READINGS:
    for lowset in ["engaged", "all"]:
        co = dreadd_corrs(L, lowset)
        fig4f[(L, lowset)] = co
        print(f"--- {L}, RE low {lowset} laps")
        print("   place cells:", {k: (len(v), "within" if abs(len(v) - EXP4F[k][0]) <= EXP4F[k][1] else "outside") for k, v in co.items()},
              "| mean r:", {k: round(v["r"].mean(), 3) for k, v in co.items()})
        for drug in ["DCZ", "CNO"]:
            ks = stats.ks_2samp(co[drug]["r"].dropna(), co["saline"]["r"].dropna())
            print(f"   {drug} vs saline overall: mean {co[drug]['r'].mean():.3f} vs {co['saline']['r'].mean():.3f}, KS D={ks.statistic:.3f} p={ks.pvalue:.2e}")
        for g in ["start", "middle", "end"]:
            sel = {k: v[group_of(v["com"].values) == g]["r"].dropna() for k, v in co.items()}
            line = []
            for k in ["saline", "DCZ", "CNO"]:
                ks = stats.ks_2samp(sel["R/RE low"], sel[k])
                line.append(f"R/RE low vs {k} KS p={ks.pvalue:.1e}")
            ks2 = {k: stats.ks_2samp(sel[k], sel["saline"]).pvalue for k in ["DCZ", "CNO"]}
            print(f"   {g:6s}: mean r " + ", ".join(f"{k} {v.mean():.3f} (n={len(v)})" for k, v in sel.items()) + " | " + "; ".join(line)
                  + f"; DCZ vs saline p={ks2['DCZ']:.1e}, CNO vs saline p={ks2['CNO']:.1e}")
print("per mouse mean r (L1, engaged):")
print(pd.DataFrame({k: v.groupby("mouse")["r"].mean() for k, v in fig4f[("L1", "engaged")].items()}).round(3).to_string())

**Result: partially reproduced.** Spatial correlation falls more after DCZ (0.657) and CNO (0.653) than after saline (0.793; KS p = 1.9e-16 and 3.3e-12), and at every position saline stays above R/RE low and DCZ and CNO lie below saline, under every reading. The place cell counts (610, 676, 842 and 641) are outside the tolerances of 928, 1139, 1629 and 1498; the dandiset holds 4 of the 6 mice.

## supp18-saline-slow-laps
> Splitting saline data into fast (before) and slow (after) laps gives only a small drop in spatial correlation, much smaller than DCZ or CNO, and no change in reliability (0.51 against 0.45) or out/in ratio (0.09 against 0.09).

Percentiles are taken within each mouse, or over the laps of all 4 mice pooled (pooled thresholds printed below); both are run, as the criterion asks. The split correlation uses fields defined on the fast laps; the same fields against all after-saline laps are shown so that the drop can be read like for like.

In [ ]:
for fast, slow in [("Sal_fast", "Sal_slow_a"), ("Sal_fast", "Sal_slow_b"), ("Sal_fast_pool", "Sal_slow_a_pool"), ("Sal_fast_pool", "Sal_slow_b_pool")]:
    sp_f = np.concatenate([K.lap_speeds(RES[s]["beh"]["conds"]["BeforeSaline"])[RES[s]["fields_laps"][fast]] for s in DR])
    sp_s = np.concatenate([K.lap_speeds(RES[s]["beh"]["conds"]["AfterSaline"])[RES[s]["fields_laps"][slow]] for s in DR])
    pf_f = pd.concat([fields_df(s, fast) for s in DR])
    pf_s = pd.concat([fields_df(s, slow) for s in DR])
    print(f"--- {fast} against {slow} ({'below' if '_a' in slow else 'above'} the 40th percentile, {'pooled over mice' if 'pool' in slow else 'within each mouse'})")
    if 'pool' in slow:
        print('   pooled thresholds (cm/s):', K.pooled_saline_speed_thresholds())
    print(f"   lap speed fast {np.nanmean(sp_f):.2f} (paper 47.30 +/- 2.61), after {np.nanmean(sp_s):.2f} (paper 35.03 +/- 1.57) cm/s")
    for p, ex in [("reliability", (0.51, 0.06, 0.45, 0.13)), ("out_in", (0.09, 0.05, 0.09, 0.05))]:
        for reading in ["per-mouse means", "pooled fields"]:
            if reading == "per-mouse means":
                a, b2 = mean_ci(pf_f.groupby("mouse")[p].mean()), mean_ci(pf_s.groupby("mouse")[p].mean())
            else:
                a, b2 = mean_ci(pf_f[p]), mean_ci(pf_s[p])
            print(f"   {p} {reading}: fast {a[0]:.3f} +/- {a[1]:.3f}, after {b2[0]:.3f} +/- {b2[1]:.3f}; within tolerance {abs(a[0] - ex[0]) <= ex[1] and abs(b2[0] - ex[2]) <= ex[3]}")
    corr_split = pd.concat([pair_corr(s, fast, "AfterSaline", RES[s]["fields_laps"][slow]) for s in DR if RES[s]["fields"].get(fast) and len(RES[s]["fields_laps"][slow])])["r"].mean()
    co = fig4f[("L1", "engaged")]
    drop_split = co["saline"]["r"].mean() - corr_split
    # the same fields (defined on the fast laps) against all after-saline laps, so that the split is compared like for like
    corr_fast_all = pd.concat([pair_corr(s, fast, "AfterSaline") for s in DR if RES[s]["fields"].get(fast)])["r"].mean()
    print(f"   fast-lap fields against all after-saline laps {corr_fast_all:.3f}; drop from it to the slow laps {corr_fast_all - corr_split:.3f}")
    drop_dcz = co["saline"]["r"].mean() - co["DCZ"]["r"].mean()
    drop_cno = co["saline"]["r"].mean() - co["CNO"]["r"].mean()
    print(f"   spatial correlation: saline pair {co['saline']['r'].mean():.3f}, fast vs slow split {corr_split:.3f}, DCZ pair {co['DCZ']['r'].mean():.3f}, CNO pair {co['CNO']['r'].mean():.3f}")
    print(f"   drop of the split {drop_split:.3f}; drop for DCZ {drop_dcz:.3f}, CNO {drop_cno:.3f}; split drop below half the DCZ drop: {drop_split < 0.5 * drop_dcz}")
    print("   fields: fast", len(pf_f), "after", len(pf_s), "| laps per mouse fast", [len(RES[s]["fields_laps"][fast]) for s in DR], "after", [len(RES[s]["fields_laps"][slow]) for s in DR])

**Result: not reproduced.** Within each mouse, the fast laps run at 30.63 cm/s (paper 47.30) and the slow laps at 21.17 cm/s (reading a; paper 35.03); with pooled percentiles 32.56 and 18.36 cm/s. Reliability is outside tolerance under all four splits (fast 0.340 to 0.438 against 0.51 +/- 0.06). The drop in spatial correlation for the split exceeds half the DCZ drop of 0.136 (0.068) under three splits (0.217, 0.141, 0.520) and is below it only for the pooled reading (b) (-0.019), where the fast laps come almost all from 2 mice. Measured from the same fast-lap fields against all after-saline laps, the drop to the slow laps is 0.086 (reading a) and 0.010 (reading b) within each mouse.

## fig4h-dreadd-reliability-outin
> DCZ and CNO lower place field reliability and raise out/in field firing, as RE low does, while saline does not.

In [ ]:
EXP4H = {"reliability": {"Rewarded": (0.41, 0.11), "RE low": (0.29, 0.12), "BeforeDCZ": (0.48, 0.10), "AfterDCZ": (0.41, 0.10), "BeforeCNO": (0.42, 0.11),
                         "AfterCNO": (0.35, 0.12), "BeforeSaline": (0.45, 0.11), "AfterSaline": (0.46, 0.08)},
         "out_in": {"Rewarded": (0.10, 0.04), "RE low": (0.17, 0.12), "BeforeDCZ": (0.10, 0.05), "AfterDCZ": (0.12, 0.04), "BeforeCNO": (0.10, 0.05),
                    "AfterCNO": (0.12, 0.05), "BeforeSaline": (0.09, 0.05), "AfterSaline": (0.09, 0.05)}}

def cdf_bins_test(a, b2, nb=20):
    edges = np.linspace(min(a.min(), b2.min()), max(a.max(), b2.max()), nb + 1)
    ca = np.cumsum(np.histogram(a, edges)[0]) / len(a)
    cb = np.cumsum(np.histogram(b2, edges)[0]) / len(b2)
    return stats.ttest_rel(ca, cb).pvalue

for L in K.LICK_READINGS:
    for lowset in ["engaged", "all"]:
        name = f"RElow_eng_{L}" if lowset == "engaged" else f"RElow_all_{L}"
        pf = {n: pd.concat([fields_df(s, n) for s in DR]) for n in ["Rewarded", "BeforeDCZ", "AfterDCZ", "BeforeCNO", "AfterCNO", "BeforeSaline", "AfterSaline"]}
        pf["RE low"] = pd.concat([fields_df(s, name) for s in DR])
        for reading in ["per-mouse means", "pooled fields"]:
            n_in = 0
            vals = {}
            for p, ex in EXP4H.items():
                for n, (v, tol) in ex.items():
                    m = mean_ci(pf[n].groupby("mouse")[p].mean())[0] if reading == "per-mouse means" else pf[n][p].mean()
                    vals[(p, n)] = m
                    n_in += abs(m - v) <= tol
            dirs = {d: (vals[("reliability", "After" + d)] < vals[("reliability", "Before" + d)], vals[("out_in", "After" + d)] > vals[("out_in", "Before" + d)]) for d in ["DCZ", "CNO"]}
            print(f"{L} RE low {lowset}, {reading}: within tolerance {n_in} of 16; DCZ/CNO (reliability lower, out/in higher) after: {dirs}")
            print("    " + "; ".join(f"{p[:3]} {n} {m:.3f}" for (p, n), m in vals.items()))
for p in ["reliability", "out_in"]:
    for d in ["Saline", "DCZ", "CNO"]:
        bm = [fields_df(s, "Before" + d)[p].mean() for s in DR]
        am = [fields_df(s, "After" + d)[p].mean() for s in DR]
        t = stats.ttest_rel(bm, am)
        a = pd.concat([fields_df(s, "Before" + d) for s in DR])[p].dropna().values
        b2 = pd.concat([fields_df(s, "After" + d) for s in DR])[p].dropna().values
        print(f"{p} {d}: per-mouse before {np.round(bm, 3).tolist()} after {np.round(am, 3).tolist()}; paired t over 4 mice p={t.pvalue:.3f}; paired t over cumulative histogram bins p={cdf_bins_test(a, b2):.3g}")

**Result: not reproduced.** All 16 means are within tolerance under both readings. Out/in firing rises after DCZ and CNO, and reliability falls after DCZ (0.452 to 0.438 per mouse) but not after CNO (0.414 to 0.417) over per-mouse means; over pooled fields both fall. Saline is not without change: the out/in ratio falls after saline (paired t over 4 mice p = 0.013) and reliability differs over the cumulative histogram bins (p = 4.5e-06), so the absence of a saline effect, which the claim states, fails under both pairings.

## fig4m-dreadd-overrepresentation
> DCZ and CNO tend to reduce place field over-representation of the reward site compared with saline, but not significantly (P = 0.06), and less than RE low does.

In [ ]:
EXP4M = {"BeforeSaline": 113, "AfterSaline": 114, "BeforeDCZ": 113, "AfterDCZ": 106, "BeforeCNO": 113, "AfterCNO": 109, "Rewarded": 110, "RE low": 103}
for L in K.LICK_READINGS:
    for lowset in ["engaged", "all"]:
        name = f"RElow_eng_{L}" if lowset == "engaged" else f"RElow_all_{L}"
        em = {}
        for s in DR:
            row = {n: end_minus_middle(fields_df(s, n)) for n in ["BeforeSaline", "AfterSaline", "BeforeDCZ", "AfterDCZ", "BeforeCNO", "AfterCNO", "Rewarded"]}
            row["RE low"] = end_minus_middle(fields_df(s, name))
            em[s] = row
        em = pd.DataFrame(em).T
        ch = pd.DataFrame({"saline": em["AfterSaline"] - em["BeforeSaline"], "DCZ": em["AfterDCZ"] - em["BeforeDCZ"],
                           "CNO": em["AfterCNO"] - em["BeforeCNO"], "RE low": em["RE low"] - em["Rewarded"]})
        t = stats.ttest_rel(ch["DCZ"], ch["saline"])
        w = stats.wilcoxon(ch["CNO"], ch["saline"])
        print(f"--- {L}, RE low {lowset}: change in end minus middle per mouse:\n{ch.round(1).to_string()}")
        print(f"    mean change saline {ch['saline'].mean():.1f}, DCZ {ch['DCZ'].mean():.1f}, CNO {ch['CNO'].mean():.1f}, RE low {ch['RE low'].mean():.1f}; "
              f"DCZ vs saline paired t p={t.pvalue:.3f}; CNO vs saline Wilcoxon p={w.pvalue:.3f}; |RE low| > |DCZ|: {abs(ch['RE low'].mean()) > abs(ch['DCZ'].mean())}")
pfc = {n: pd.concat([fields_df(s, n) for s in DR])["com"].dropna() for n in ["BeforeSaline", "AfterSaline", "BeforeDCZ", "AfterDCZ", "BeforeCNO", "AfterCNO", "Rewarded"]}
pfc["RE low"] = pd.concat([fields_df(s, "RElow_eng_L1") for s in DR])["com"].dropna().astype(float)
pfc["RE low (all laps)"] = pd.concat([fields_df(s, "RElow_all_L1") for s in DR])["com"].dropna().astype(float)
EXP4M["RE low (all laps)"] = EXP4M["RE low"]
fits = {n: stats.norm.fit(v) for n, v in pfc.items() if len(v) > 1}
print("fields per set:", {n: len(v) for n, v in pfc.items()})
print("Gaussian fits mean +/- SD (cm):", {n: tuple(np.round(f, 1)) for n, f in fits.items()}, "| within 10% of the paper's means:",
      {n: abs(f[0] - EXP4M[n]) <= 0.1 * EXP4M[n] for n, f in fits.items()})

**Result: reproduced.** The change in end minus middle field percentage is 2.1 points after DCZ against 4.3 after saline (paired t p = 0.402, not significant), and the RE low change (-11.9 under L1, engaged laps, 3 mice; 009 has 2 engaged RE low laps) is larger in size; CNO against saline Wilcoxon p = 0.625. DCZ does not reduce the over-representation in absolute terms here; it rises less than after saline, and CNO (5.8) rises more than saline. The test has 4 mice. All Gaussian means are within 10% of the paper's.

## Blocked claims

- supp2-decoder-stable-control and controls-no-change-over-time: the control mice that stayed in R for the whole session are not in the dandiset (its 14 files are the 10 extinction mice and the 4 DREADD mice).
- supp6-pupil-engagement: no pupil video or pupil traces are in the files (every HDF5 object of every file is listed in the data check; `processing/ophys/` and `analysis/` are empty).
- supp15-tdtomato-controls: the three DAT-Cre tdTomato mice are not in the dandiset; all four DREADD files are described in full as "Dat-Cre mice injected with inhibitory DREADD receptor in VTA-Dopmaine neurons" (data check).
- fig5-da-axon-ramps, fig5k-ramp-slope-max, fig5l-ramps-return-rr, supp20-novel-axons, supp21-heterogeneous-axons: the VTA axon imaging is not in the dandiset.
- supp14-reward-cells: the reward cell definition is "as described in ref. 24": a cell that "fired at the reward zone on the track (40 cm before reward) and around reward delivery (2 seconds before and after)". Neither the paper nor the claims file says what counts as firing there (a place field with its centre in the zone, a peak, a threshold on activity), and the criterion is not in the paper's code. Any threshold chosen here would be arbitrary, and the count of 43 cells is the claim's main number. The on-track part could be read from the place fields found here (R and novel fields with a COM in the last 40 cm, 5 mice), but the criterion also needs the within-R reading S1, which needs the control mice, and readings S1 and S2 must agree; it is not run.


## Summary

| Claim | Outcome |
|---|---|
| session-lap-counts | reproduced |
| lick-persistence-ur | not reproduced |
| fig1d-decoder-drop-ur | reproduced |
| fig1e-decoder-follows-licking | reproduced |
| fig1f-decoder-r2 | partially reproduced |
| fig1g-decoder-error-all-locations | reproduced |
| supp4-lickstop-definition-robust | reproduced |
| supp4-lapwise-r2 | partially reproduced |
| supp2-decoder-stable-control | blocked (data missing) |
| supp5-not-running-speed | partially reproduced |
| approach-ratio | not reproduced |
| engaged-lap-counts | not reproduced |
| supp6b-decoder-engaged-laps | reproduced |
| disengaged-laps-uniform | reproduced |
| supp6-pupil-engagement | blocked (data missing) |
| fig2a-pv-cluster-transition | not reproduced |
| fig2c-spatial-correlation-drop | partially reproduced |
| fig2d-remapping-all-locations | partially reproduced |
| fig2e-place-field-fates | not reproduced |
| fig3a-field-numbers-similar | not reproduced |
| fig3b-reliability | reproduced |
| fig3b-out-in-ratio | not reproduced |
| fig3b-width-intensity | not reproduced |
| fig3c-degradation-all-locations | partially reproduced |
| supp11-all-mice-parameters | not reproduced |
| fig3d-reward-overrepresentation | not reproduced |
| reward-zone-field-turnover | not reproduced |
| controls-no-change-over-time | blocked (data missing) |
| supp13-novel-environment | partially reproduced |
| supp14-reward-cells | blocked (methods underspecified) |
| supp16-dreadd-running-speed | not reproduced |
| fig4f-dreadd-spatial-correlation | partially reproduced |
| supp18-saline-slow-laps | not reproduced |
| fig4h-dreadd-reliability-outin | not reproduced |
| fig4m-dreadd-overrepresentation | reproduced |
| supp15-tdtomato-controls | blocked (data missing) |
| fig5-da-axon-ramps, fig5k, fig5l, supp20, supp21 | blocked (data missing) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's.

- The dandiset holds 10 of the 12 extinction mice (5 of the 6 with RR and a novel session) and 4 of the 6 DREADD mice (4 of 5 with saline). Counts of laps, fields and cells are over these mice. Thresholds written as counts of mice are applied as the same fraction of the mice here.
- The files hold raw fluorescence, not the paper's dF/F. dF/F here uses the 8th percentile in a 15 s window, and significant transients start above 2 and end below 0.5 robust SD and last 0.5 s or more (ref. 75; assumption). Place field candidates are bins above 25% of the way from baseline to peak (ref. 75), and the bootstrap uses 100 circular shifts. The fields found are wider than the paper's (70 to 90 cm in R against 53 cm); the candidate regions are 31.4 to 43.2 cm wide (per-mouse means), narrower than the paper's 53 to 61 cm, and the Methods width rule widens them.
- The DREADD files carry a constant offset of about 17000 counts, removed as the file minimum before dF/F (assumption).
- Position is in volts; the top of the range, where reward is delivered, is read as 200 cm. The 30 Hz clock agrees with the 1.5 s reward pause (46 to 49 frames), yet lap times are longer and speeds slower than the paper's (wild-type R laps 12.2 s against 7.34 s; DREADD lap speeds about 26 against 42 cm/s). Lap time does not depend on the volts-to-cm scale, so the difference lies in the data or in a definition of lap time the paper does not give.
- The decoder treats the track as a circle by taking each prediction at its nearest copy of the true bin, as the criterion's "track treated as circular" says; without it R2 values are far lower (shown).
- The approach ratio that defines engaged laps is taken over the traversal of the track, from the lap start to the track end, leaving out the pause after the reward (Methods, "velocity was calculated at each point along the 2 m track"; assumption); laps with an undefined ratio are neither engaged nor disengaged. With this split 46% of RE low laps are engaged and 2 of the 5 R-UR-RR mice have more than 12 engaged laps; the code's ratio, a variant, gives 67% and 4 mice. The whole-lap ratio of the first version took in the pause and is shown as a variant only.
- The reward zone starts at the mean position of the first R lick before the reward and after 20 cm (licks just after the teleport belong to the previous reward; the 20 cm is an assumption).
- The within-R reading S1 (control mice) of the spatial correlation could not be run, so the claims that need S1 and S2 to agree are partially reproduced at most.

What was found about the dataset.

- sub-014 has 6000 fewer behaviour rows than fluorescence frames; decoding shows the missing rows are at the start of BeforeCNO, so every later behaviour row is 6000 frames early. sub-NR6 has 8 extra behaviour rows and sub-CFC4 one row fewer, at the end.
- The DREADD fluorescence carries an undocumented offset of about 17000 counts.
- In NR6 and NR14 a reward is delivered at the end of the first UR lap as the frame counts define the conditions.
- Conditions are not marked in the files except as frame counts in a free-text description; the position channel has no unit or scale; the files do not say which mice the paper's figures use.
- The control mice, the dark sessions, the pupil video, the tdTomato mice and the VTA axon imaging are not in the dandiset, and 2 extinction mice and 2 DREADD mice are missing.
